# Axion PINN — Advanced: WKB Envelope + Multi-scale Fourier + Sequential Windows

## ปัญหาและแนวทางแก้

### ปัญหาหลัก: Spectral Bias
Neural networks มี **spectral bias** — เรียนรู้ความถี่ต่ำก่อนเสมอ  
สำหรับ axion ที่มวลสูง ($m \gg H_0$) จะมี oscillation ถี่ถึง $\sim 10^5$–$10^6$ ครั้งใน $t\in[0,1]$  
แม้ใช้ Fourier embedding และ rescaling แล้ว ก็ยังยากที่จะ resolve

---

### สามเทคนิคที่ใช้ร่วมกัน

#### 1. WKB Envelope Decomposition ⭐ (สำคัญที่สุด)
แทนที่จะให้ network เรียน $\phi(t)$ (ซึ่งสั่นถี่มาก) เปลี่ยนเป็นเรียน **envelope** $A(t)$ และ **phase correction** $\delta\Phi(t)$ ซึ่งเปลี่ยนแปลงช้ามาก:

$$\phi(t) = \underbrace{A(t)}_{\text{smooth, network เรียน}} \cdot \cos\!\Big(\underbrace{m_\mathrm{sim}\, t}_{\text{analytic fast part}} + \underbrace{\delta\Phi(t)}_{\text{smooth correction}}\Big)$$

- Network output $A(t)$ และ $\delta\Phi(t)$ เปลี่ยนแปลงช้า $\sim \mathcal{O}(H/m)$ → ง่ายมากสำหรับ NN
- High-frequency oscillation จับโดย $\cos(m_\mathrm{sim}\,t)$ แบบ analytic → ไม่ต้องให้ network เรียน
- Hard IC: $A(0) = \phi_0$, $A'(0) = \dot\phi_0$, $\delta\Phi(0) = 0$

#### 2. Multi-scale Fourier Embedding ⭐⭐
แทนที่จะใช้ scale เดียว ใช้หลาย frequency bands พร้อมกัน (log-spaced จาก 1 ถึง $m_\mathrm{sim}$)
แต่ละ band มี trainable log-scale → ครอบ spectrum ทั้งหมดอัตโนมัติ

$$\text{embed}(t) = \big[\underbrace{\sin(2\pi \cdot s_1 \cdot t), \cos(\cdots)}_{\text{low band}},\ \underbrace{\cdots}_{\text{mid band}},\ \underbrace{\sin(2\pi \cdot s_k \cdot t), \cos(\cdots)}_{\text{high band}}\big]$$

#### 3. Sequential Window Training ⭐⭐
แบ่ง $[0, T]$ เป็น $K$ windows สั้นๆ แต่ละ window มี oscillation น้อยลง $K$ เท่า
และ error ไม่สะสมข้ามทั้ง domain:
$$[0, T] \to [0,\Delta t],\ [\Delta t, 2\Delta t],\ \ldots$$
- IC ของแต่ละ window = prediction จาก window ก่อนหน้า (รวม $\dot\phi$!)
- ใช้ `FullICFieldNet`: $\phi(\tau) = \phi_0 + \dot\phi_0\,\tau + \tau^2 \cdot \mathrm{net}(\tau)$ (hard IC สำหรับทั้ง $\phi$ และ $\dot\phi$)

---

### สรุปการออกแบบ

```
AdvancedPINN  =  ScaleFactorNet(a0)  +  WKBFieldNet(φ0, φ̇0, m_sim)
                                         └─ MultiScaleEmbedding
                                         └─ SinResBlocks
                                         └─ two heads: A_raw, δΦ_raw

SequentialPINN per window  =  ScaleFactorNet(a0)  +  FullICFieldNet(φ0, φ̇0, m_sim)
                                                         └─ MultiScaleEmbedding
                                                         └─ SinResBlocks
```

## 1. Imports & Setup

In [2]:
import os, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE  = torch.float64
EPS    = 1e-20

print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

Device: cpu | PyTorch: 2.12.0+cpu


## 2. Cosmological Parameters & Mass Regime Analyzer

In [3]:
RHO_M0 = 0.81; RHO_R0 = 0.00027138; RHO_L0 = 2.19
H0_eV  = 6.582e-16 * 2.184e-18   # ℏH0 in eV

MAX_N_OSC_DIRECT = 500   # threshold for direct vs rescaling


def analyze_mass_regime(m_target: float, t_end: float = 1.0) -> dict:
    """Auto-select direct/rescaling strategy and derive sim parameters."""
    n_osc = m_target / (2 * np.pi) * t_end
    a_osc = (3 * RHO_R0 / m_target**2) ** 0.25

    if n_osc <= MAX_N_OSC_DIRECT:
        mode, m_sim, lam = 'direct', m_target, 1.0
        t_end_sim = t_end
    else:
        # Choose m_sim so that N_osc_sim ≈ MAX_N_OSC_DIRECT/2
        C      = 2 * np.pi * (MAX_N_OSC_DIRECT / 2) / t_end
        m_sim  = max(100.0, (C * m_target**0.5) ** (2.0/3.0))
        lam    = np.sqrt(m_target / m_sim)
        mode   = 'rescaling'
        t_end_sim = t_end / lam

    regime = ('slow-roll' if m_target < 3 else 'onset' if m_target < 50 else
              'oscillating' if m_target < 500 else 'rapid-osc')
    return dict(mode=mode, regime=regime, m_target=m_target, m_sim=m_sim,
                lam=lam, t_end=t_end, t_end_sim=t_end_sim,
                a_osc=a_osc, n_osc=n_osc, rho_scale=lam**2)


def print_mass_summary(mc):
    print(f"{'─'*60}")
    print(f"  m_target={mc['m_target']:.3g}  regime={mc['regime']}  mode={mc['mode']}")
    print(f"  N_osc(physical)={mc['n_osc']:.2e}  |  a_osc={mc['a_osc']:.3e}")
    if mc['mode'] == 'rescaling':
        print(f"  m_sim={mc['m_sim']:.3g}  λ={mc['lam']:.3g}  t_end_sim={mc['t_end_sim']:.3e}")
    print(f"  m_a ≈ {mc['m_target']*H0_eV:.3e} eV")
    print(f"{'─'*60}")

## 3. ODE Reference Solver

In [4]:
def _ode_rhs(t, y, m2, rho_m0, rho_r0, rho_l0):
    a, phi, phi_dot = y
    rho = 0.5*phi_dot**2 + 0.5*m2*phi**2 + rho_m0/max(a**3,1e-60) + rho_r0/max(a**4,1e-60) + rho_l0
    H = np.sqrt(max(rho/3, 0))
    return [a*H, phi_dot, -3*H*phi_dot - m2*phi]


def solve_ode(m_sim, phi0, phi_dot0=0.0, a0=1e-8, t_end_sim=1.0,
              rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
              n_points=10000) -> dict:
    """High-accuracy ODE solve in simulation coordinates."""
    m2 = m_sim**2
    t_eval = np.logspace(np.log10(max(a0, 1e-13)), np.log10(t_end_sim), n_points)
    t0 = time.time()
    sol = solve_ivp(lambda t, y: _ode_rhs(t, y, m2, rho_m0, rho_r0, rho_l0),
                    [t_eval[0], t_eval[-1]], [a0, phi0, phi_dot0],
                    t_eval=t_eval, method='DOP853', rtol=1e-10, atol=1e-12)
    print(f"ODE: {time.time()-t0:.2f}s  success={sol.success}  nfev={sol.nfev}")
    a, phi, phi_dot = sol.y
    rho_ax = 0.5*phi_dot**2 + 0.5*m2*phi**2
    return dict(t=sol.t, a=a, phi=phi, phi_dot=phi_dot,
                rho_ax=rho_ax,
                rho_m=rho_m0/a**3,
                rho_r=rho_r0/a**4,
                H=np.sqrt((rho_ax + rho_m0/a**3 + rho_r0/a**4 + rho_l0)/3.0))

## 4. Technique 1 — Multi-scale Fourier Embedding

### ปัญหาของ Single-scale Fourier Embedding
Embedding แบบเดิมใช้ scale เดียว (trainable) ซึ่งมักไปติดที่ frequency ต่ำ:
$$\text{embed}(t) = [\sin(2\pi s\,b_1 t),\ \cos(2\pi s\,b_1 t),\ \ldots]$$

### Multi-scale: ครอบ spectrum ทั้งหมด
ใช้หลาย bands พร้อมกัน แต่ละ band มี trainable log-scale ของตัวเอง:
$$\text{embed}(t) = \big[\underbrace{F_1(t)}_{\text{slow }\sim 1},\ \underbrace{F_2(t)}_{\text{mid}},\ \underbrace{F_3(t)}_{\text{fast }\sim m_\mathrm{sim}}\big]$$

จำนวน bands = $\max(2,\ \lceil\log_{10}(m_\mathrm{sim})\rceil)$ → auto-scale ตามมวล

In [5]:
class MultiScaleEmbedding(nn.Module):
    """
    Concatenated Fourier features over log-spaced frequency bands.

    For m_sim, auto-generates n_bands = max(2, ceil(log10(m_sim))) bands
    spanning log-uniformly from scale=1 up to scale=m_sim.
    Each band has its own trainable log_scale (initialised at the band centre).

    output dim = n_bands * 2 * n_per_band
    """
    def __init__(self, n_per_band: int, m_sim: float, dtype=torch.float64):
        super().__init__()
        m_sim = float(m_sim)
        n_bands = max(2, int(np.ceil(np.log10(max(m_sim, 10.0)))))

        # Log-spaced initial scales: 1 → m_sim
        log_inits = np.linspace(0.0, np.log(m_sim), n_bands)

        # Trainable log-scales, one per band
        self.log_scales = nn.Parameter(
            torch.tensor(log_inits, dtype=dtype))   # (n_bands,)

        # Random projection matrices (frozen), one per band
        B = torch.randn(n_bands, n_per_band, 1, dtype=dtype)
        self.register_buffer('B', B)   # (n_bands, n_per_band, 1)

        self.n_bands    = n_bands
        self.n_per_band = n_per_band
        self.out_dim    = n_bands * 2 * n_per_band

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t: (N, 1)
        scales = torch.exp(self.log_scales)  # (n_bands,)
        parts = []
        for b in range(self.n_bands):
            proj = 2.0 * np.pi * scales[b] * (t @ self.B[b].T)  # (N, n_per_band)
            parts.append(torch.sin(proj))
            parts.append(torch.cos(proj))
        return torch.cat(parts, dim=-1)  # (N, out_dim)


# ── Sanity check ──────────────────────────────────────────────
for m_test in [10, 100, 1000]:
    emb = MultiScaleEmbedding(16, m_test)
    n_bands = emb.n_bands
    init_scales = np.exp(emb.log_scales.detach().numpy())
    print(f"m_sim={m_test:5d}  n_bands={n_bands}  scales={np.round(init_scales, 1)}  out_dim={emb.out_dim}")

m_sim=   10  n_bands=2  scales=[ 1. 10.]  out_dim=64
m_sim=  100  n_bands=2  scales=[  1. 100.]  out_dim=64
m_sim= 1000  n_bands=3  scales=[   1.    31.6 1000. ]  out_dim=96


## 5. Technique 2 — WKB Envelope + Phase Networks

### ทำไม WKB ถึงช่วยได้มาก

ใน WKB approximation ($m \gg H$), solution ของ Klein-Gordon มีรูปแบบ:
$$\phi(t) \approx A(t)\cos\!\Big(m\,t + \delta\Phi(t)\Big), \quad \frac{\dot A}{A} \sim \frac{H}{2},\quad \dot{\delta\Phi} \sim \mathcal{O}\!\left(\frac{H^2}{m}\right)$$

ทั้ง $A$ และ $\delta\Phi$ เปลี่ยนแปลงช้ามาก (timescale $\sim 1/H$) เทียบกับ oscillation ($\sim 1/m$)

**Network เรียน $A_\mathrm{raw}(t)$ และ $\delta\Phi_\mathrm{raw}(t)$ เท่านั้น** — ซึ่งเกือบ flat!  
ส่วน $\cos(m_\mathrm{sim}\,t)$ คำนวณ analytically — ไม่มี spectral bias issue

### Hard IC ที่ enforce ได้แน่นอน
$$A(t) = \phi_0 + \dot\phi_0\,t + t^2 \cdot A_\mathrm{raw}(t) \implies \begin{cases}A(0) = \phi_0\\A'(0) = \dot\phi_0\end{cases}$$
$$\delta\Phi(t) = t \cdot \delta\Phi_\mathrm{raw}(t) \implies \delta\Phi(0) = 0$$

ดังนั้น $\phi(0) = \phi_0\cos(0) = \phi_0$ ✓ และ $\dot\phi(0) = \dot\phi_0$ ✓

In [6]:
class SinResBlock(nn.Module):
    """SIREN-style residual block — proven effective for oscillatory problems."""
    def __init__(self, width: int, dtype=torch.float64):
        super().__init__()
        self.fc1 = nn.Linear(width, width, dtype=dtype)
        self.fc2 = nn.Linear(width, width, dtype=dtype)
        nn.init.uniform_(self.fc1.weight, -np.sqrt(6/width), np.sqrt(6/width))
        nn.init.uniform_(self.fc2.weight, -np.sqrt(6/width), np.sqrt(6/width))

    def forward(self, x):
        return x + self.fc2(torch.sin(self.fc1(x)))


class ScaleFactorNet(nn.Module):
    """
    Network for a(t) with hard IC:  a(t) = a0 + t·softplus(net(t))
    Guarantees a(0)=a0 and a is monotonically non-decreasing.
    """
    def __init__(self, a0: float, n_per_band: int = 16, hidden: int = 64,
                 depth: int = 3, dtype=torch.float64):
        super().__init__()
        # Small m_sim for a(t): a varies slowly, use a gentle embedding scale
        self.emb = MultiScaleEmbedding(n_per_band, m_sim=10.0, dtype=dtype)
        D = self.emb.out_dim
        layers = [nn.Linear(D, hidden, dtype=dtype), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden, dtype=dtype), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1, dtype=dtype))
        self.net = nn.Sequential(*layers)
        self.sp  = nn.Softplus()
        self.register_buffer('a0', torch.tensor([[a0]], dtype=dtype))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.a0 + t * self.sp(self.net(self.emb(t)))


class WKBFieldNet(nn.Module):
    """
    WKB decomposition:  φ(t) = A(t) · cos(m_sim·t + δΦ(t))

    where the slow functions are:
        A(t)    = phi0 + phi_dot0·t + t²·A_raw(t)
        δΦ(t)   = t·δΦ_raw(t)

    A_raw and δΦ_raw are nearly constant (slowly varying) → easy to learn.
    The high-frequency oscillation is encoded analytically in cos(m_sim·t + δΦ).

    Hard ICs: φ(0) = phi0 ✓,  φ̇(0) = phi_dot0 ✓  (verified analytically below)
    """
    def __init__(self, phi0: float, phi_dot0: float = 0.0,
                 m_sim: float = 100.0,
                 n_per_band: int = 20, hidden: int = 128, depth: int = 3,
                 dtype=torch.float64):
        super().__init__()
        self.m_sim = float(m_sim)
        self.register_buffer('phi0',     torch.tensor([[float(phi0)]],     dtype=dtype))
        self.register_buffer('phi_dot0', torch.tensor([[float(phi_dot0)]], dtype=dtype))

        # Use SLOW embedding bands for A and δΦ
        # (they only need to capture timescale ~1/H, not ~1/m)
        m_slow = max(1.0, m_sim**0.4)   # intermediate scale is enough
        self.emb = MultiScaleEmbedding(n_per_band, m_slow, dtype=dtype)
        D = self.emb.out_dim

        trunk = [nn.Linear(D, hidden, dtype=dtype)]
        for _ in range(depth):
            trunk.append(SinResBlock(hidden, dtype=dtype))
        self.trunk = nn.Sequential(*trunk)

        self.head_A    = nn.Linear(hidden, 1, dtype=dtype)
        self.head_dPhi = nn.Linear(hidden, 1, dtype=dtype)

    def get_envelope_phase(self, t: torch.Tensor):
        """Return (A(t), δΦ(t)) separately — useful for diagnostics."""
        h = self.trunk(self.emb(t))
        A_raw    = self.head_A(h)
        dPhi_raw = self.head_dPhi(h)
        A    = self.phi0 + self.phi_dot0 * t + t**2 * A_raw   # A(0)=phi0, A'(0)=phi_dot0
        dPhi = t * dPhi_raw                                    # δΦ(0)=0
        return A, dPhi

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        A, dPhi = self.get_envelope_phase(t)
        return A * torch.cos(self.m_sim * t + dPhi)


class FullICFieldNet(nn.Module):
    """
    Standard parametrisation with BOTH phi0 AND phi_dot0 hard constraints:
        φ(t) = phi0 + phi_dot0·t + t²·net(t)

    Hard ICs: φ(0)=phi0 ✓,  φ̇(0)=phi_dot0 ✓

    Uses MultiScaleEmbedding for full frequency coverage.
    Designed for sequential windows (each window has manageable N_osc
    so WKB cos decomposition is not needed — just good embeddings + full IC).
    """
    def __init__(self, phi0: float, phi_dot0: float = 0.0,
                 m_sim: float = 100.0,
                 n_per_band: int = 20, hidden: int = 128, depth: int = 4,
                 dtype=torch.float64):
        super().__init__()
        self.register_buffer('phi0',     torch.tensor([[float(phi0)]],     dtype=dtype))
        self.register_buffer('phi_dot0', torch.tensor([[float(phi_dot0)]], dtype=dtype))

        self.emb  = MultiScaleEmbedding(n_per_band, m_sim, dtype=dtype)
        D = self.emb.out_dim
        layers = [nn.Linear(D, hidden, dtype=dtype)]
        for _ in range(depth):
            layers.append(SinResBlock(hidden, dtype=dtype))
        self.net  = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1, dtype=dtype)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.phi0 + self.phi_dot0 * t + t**2 * self.head(self.net(self.emb(t)))


class GeneralPINN(nn.Module):
    """Wraps any (ScaleFactorNet, field_net) pair into a single model."""
    def __init__(self, a_net: ScaleFactorNet, phi_net):
        super().__init__()
        self.a_net   = a_net
        self.phi_net = phi_net

    def forward(self, t: torch.Tensor):
        return self.a_net(t), self.phi_net(t)


def build_advanced_pinn(mc: dict, phi0: float, phi_dot0: float = 0.0,
                        a0: float = 1e-8,
                        n_per_band_a: int = 16, n_per_band_phi: int = 20,
                        hidden_a: int = 64, hidden_phi: int = 128,
                        depth_a: int = 3, depth_phi: int = 3,
                        dtype=torch.float64) -> GeneralPINN:
    """Build WKB-based AdvancedPINN for a given mass config."""
    a_net   = ScaleFactorNet(a0, n_per_band_a, hidden_a, depth_a, dtype=dtype)
    phi_net = WKBFieldNet(phi0, phi_dot0, mc['m_sim'],
                          n_per_band_phi, hidden_phi, depth_phi, dtype=dtype)
    return GeneralPINN(a_net, phi_net)


def build_window_pinn(a0: float, phi0: float, phi_dot0: float,
                      m_sim: float,
                      n_per_band_a: int = 16, n_per_band_phi: int = 20,
                      hidden_a: int = 64, hidden_phi: int = 128,
                      depth_a: int = 3, depth_phi: int = 4,
                      dtype=torch.float64) -> GeneralPINN:
    """Build window PINN (FullICFieldNet) for sequential training."""
    a_net   = ScaleFactorNet(a0, n_per_band_a, hidden_a, depth_a, dtype=dtype)
    phi_net = FullICFieldNet(phi0, phi_dot0, m_sim,
                             n_per_band_phi, hidden_phi, depth_phi, dtype=dtype)
    return GeneralPINN(a_net, phi_net)


# ── IC verification ───────────────────────────────────────────
for _phi0, _pdot0, _m in [(1.0, 0.0, 200.0), (-0.5, 0.3, 100.0)]:
    _wkb = WKBFieldNet(_phi0, _pdot0, _m)
    _fic = FullICFieldNet(_phi0, _pdot0, _m)
    _t0  = torch.zeros(1, 1, dtype=torch.float64, requires_grad=True)
    _pw  = _wkb(_t0);  _pdw = torch.autograd.grad(_pw.sum(), _t0)[0]
    _pf  = _fic(_t0);  _pdf = torch.autograd.grad(_pf.sum(), _t0)[0]
    print(f"WKB:  φ(0)={_pw.item():.4f}(want {_phi0})  φ'(0)={_pdw.item():.4f}(want {_pdot0})")
    print(f"FullIC: φ(0)={_pf.item():.4f}(want {_phi0})  φ'(0)={_pdf.item():.4f}(want {_pdot0})")

WKB:  φ(0)=1.0000(want 1.0)  φ'(0)=0.0000(want 0.0)
FullIC: φ(0)=1.0000(want 1.0)  φ'(0)=0.0000(want 0.0)
WKB:  φ(0)=-0.5000(want -0.5)  φ'(0)=0.3000(want 0.3)
FullIC: φ(0)=-0.5000(want -0.5)  φ'(0)=0.3000(want 0.3)


## 6. Physics Loss

Loss function เหมือนเดิม — autograd ผ่าน $\phi(t)$ ไม่ว่าจะ parametrize แบบไหน  
สำหรับ WKB: leading-order cancellation ระหว่าง $\phi''$ และ $m^2\phi$ ทำให้ residual เป็น order $H/m$ โดยอัตโนมัติ

In [7]:
# ── Numerical constants ───────────────────────────────────────────────────────
_INV_SQRT3 = 3.0**(-0.5)   # 1/√3
_SQRT3     = 3.0**( 0.5)   #  √3


def physics_residuals(model: GeneralPINN, t: torch.Tensor,
                      m_sim: float,
                      rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0):
    """
    Residuals directly from the two governing equations:

    Friedmann (eq. 1):
        ȧ - (1/√3) * √(½a²φ̇² + ½m²a²φ² + ρ_m/a + ρ_r/a² + ρ_λ a²) = 0

    Klein-Gordon (eq. 2):
        φ̈ + √3 * √(½φ̇² + ½m²φ² + ρ_m/a³ + ρ_r/a⁴ + ρ_λ) * φ̇ + m²φ = 0

    Both residuals are normalised by their own natural scale (detached)
    so the two loss components stay numerically comparable during training.

    Returns: R_F (N,1), R_KG (N,1), H (N,1)
    """
    t = t.requires_grad_(True)
    a, phi = model(t)

    a_t    = torch.autograd.grad(a,    t, torch.ones_like(a),    create_graph=True)[0]
    phi_t  = torch.autograd.grad(phi,  t, torch.ones_like(phi),  create_graph=True)[0]
    phi_tt = torch.autograd.grad(phi_t, t, torch.ones_like(phi_t), create_graph=True)[0]

    m2 = float(m_sim) ** 2

    # ── powers of a (clamped for numerical stability) ─────────────────────────
    inv_a  = 1.0 / torch.clamp(a, min=EPS)
    inv_a2 = inv_a  * inv_a
    inv_a3 = inv_a2 * inv_a
    inv_a4 = inv_a3 * inv_a

    # ── Friedmann residual  (eq. 1) ───────────────────────────────────────────
    #   inner_F = ½a²φ̇² + ½m²a²φ² + ρ_m/a + ρ_r/a² + ρ_λ a²
    inner_F = (0.5 * a**2 * phi_t**2
             + 0.5 * m2   * a**2 * phi**2
             + rho_m0 * inv_a
             + rho_r0 * inv_a2
             + rho_l0 * a**2)
    rhs_F = _INV_SQRT3 * torch.sqrt(torch.clamp(inner_F, min=EPS))
    R_F   = a_t - rhs_F

    # ── Klein-Gordon residual  (eq. 2) ────────────────────────────────────────
    #   inner_KG = ½φ̇² + ½m²φ² + ρ_m/a³ + ρ_r/a⁴ + ρ_λ
    inner_KG = (0.5 * phi_t**2
              + 0.5 * m2 * phi**2
              + rho_m0 * inv_a3
              + rho_r0 * inv_a4
              + rho_l0)
    H3   = _SQRT3 * torch.sqrt(torch.clamp(inner_KG, min=EPS))
    R_KG = phi_tt + H3 * phi_t + m2 * phi

    # ── H (Hubble) for diagnostics ────────────────────────────────────────────
    #   H = ȧ/a  ≡  rhs_F / a  (exact when R_F = 0)
    H = rhs_F.detach() / (a.detach() + EPS)

    # ── Normalise for comparable loss scales (detached — no gradient impact) ──
    scale_F  = torch.abs(rhs_F).detach() + EPS         # ~ |ȧ_true|
    scale_KG = (m2 * torch.abs(phi).detach()
              + torch.abs(phi_tt).detach()) + EPS       # ~ m²|φ|

    return R_F / scale_F, R_KG / scale_KG, H


def causal_weight(t: torch.Tensor, eps: float = 0.5) -> torch.Tensor:
    return torch.exp(-eps * t / (t.max() + EPS)).detach()


class AdaptiveWeights:
    def __init__(self, w_F=1.0, w_KG=1.0, update_every=200, alpha=0.9):
        self.w_F = w_F; self.w_KG = w_KG
        self.alpha = alpha; self.update_every = update_every; self._n = 0

    def step(self, lF, lKG):
        self._n += 1
        if self._n % self.update_every == 0:
            mean = (lF + lKG) / 2.0 + EPS
            self.w_F  = self.alpha*self.w_F  + (1-self.alpha)*mean/(lF  + EPS)
            self.w_KG = self.alpha*self.w_KG + (1-self.alpha)*mean/(lKG + EPS)

    def get(self):
        return self.w_F, self.w_KG


def total_loss(model, t, m_sim, rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
               w_F=1.0, w_KG=1.0, causal_eps=0.5):
    """
    L = w_F * <(R_F / scale_F)^2>  +  w_KG * <(R_KG / scale_KG)^2>

    R_F  = exact LHS of Friedmann eq. 1 (should be 0)
    R_KG = exact LHS of Klein-Gordon eq. 2 (should be 0)
    """
    R_F, R_KG, H = physics_residuals(model, t, m_sim, rho_m0, rho_r0, rho_l0)
    cw  = causal_weight(t, causal_eps)
    lF  = w_F  * torch.mean(cw * R_F**2)
    lKG = w_KG * torch.mean(cw * R_KG**2)
    return lF + lKG, lF.item(), lKG.item()


## 7. Training Pipeline (Adam → L-BFGS)

In [8]:
def _sample_collocation(n: int, t_min: float, t_max: float,
                        a_osc: float, dtype, device) -> torch.Tensor:
    """Adaptive sampling: 50% log, 30% cluster near a_osc, 20% uniform."""
    n_log = int(n * 0.50); n_cl = int(n * 0.30); n_uni = n - n_log - n_cl
    t_safe = max(t_min, 1e-14)

    t_log = np.logspace(np.log10(t_safe), np.log10(t_max), n_log)
    t_center = float(a_osc * t_max / max(t_max, 1e-20))
    t_center = np.clip(t_center, t_safe, t_max * 0.9)
    t_cl  = np.clip(np.abs(np.random.normal(t_center, t_center*0.4, n_cl)), t_safe, t_max)
    t_uni = np.random.uniform(t_safe, t_max, n_uni)
    t_all = np.sort(np.concatenate([t_log, t_cl, t_uni]))
    return torch.tensor(t_all.reshape(-1, 1), dtype=dtype, device=device)


def train_single(model: GeneralPINN, mc: dict,
                 n_colloc=4000, batch_size=512,
                 adam_epochs=8000, adam_lr=5e-4,
                 lbfgs_iters=2000,
                 causal_eps=0.5, print_every=1000,
                 rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
                 device=DEVICE, dtype=DTYPE) -> dict:
    """
    Full Adam→L-BFGS training for a single-window model.
    Works for both AdvancedPINN (WKB) and window PINN (FullIC).
    """
    m_sim     = mc['m_sim']
    t_end_sim = mc['t_end_sim']
    a_osc     = mc['a_osc']

    model.train()
    hist = dict(adam_loss=[], lF=[], lKG=[], w_F=[], w_KG=[], lbfgs_loss=[])
    aw = AdaptiveWeights(update_every=200)

    # ── Adam ──────────────────────────────────────────────────
    opt = optim.Adam(model.parameters(), lr=adam_lr, eps=1e-9)
    sched = CosineAnnealingLR(opt, T_max=adam_epochs, eta_min=adam_lr*1e-3)
    t0 = time.time()
    print(f"  Adam  m_sim={m_sim:.3g}  t_end={t_end_sim:.3e}  [{mc['mode']}]")

    for epoch in range(adam_epochs):
        t_all = _sample_collocation(n_colloc, 0.0, t_end_sim, a_osc, dtype, device)
        idx   = torch.randperm(n_colloc, device=device)[:batch_size]
        t_b   = t_all[idx].clone()

        w_F, w_KG = aw.get()
        opt.zero_grad()
        loss, lF, lKG = total_loss(model, t_b, m_sim, rho_m0, rho_r0, rho_l0,
                                    w_F=w_F, w_KG=w_KG, causal_eps=causal_eps)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); aw.step(lF, lKG)

        hist['adam_loss'].append(loss.item())
        hist['lF'].append(lF); hist['lKG'].append(lKG)
        hist['w_F'].append(w_F); hist['w_KG'].append(w_KG)

        if epoch % print_every == 0 or epoch == adam_epochs - 1:
            print(f"    [{epoch:5d}/{adam_epochs}] loss={loss.item():.3e}  "
                  f"lF={lF:.3e}  lKG={lKG:.3e}  lr={opt.param_groups[0]['lr']:.2e}")

    print(f"  Adam done {time.time()-t0:.1f}s")

    # ── L-BFGS ────────────────────────────────────────────────
    print("  L-BFGS refinement ...")
    t_full = _sample_collocation(n_colloc, 0.0, t_end_sim, a_osc, dtype, device)
    t_full.requires_grad_(True)
    w_F, w_KG = aw.get()
    lb_buf = []

    opt_lb = optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iters,
                          tolerance_grad=1e-11, tolerance_change=1e-12,
                          history_size=150, line_search_fn='strong_wolfe')

    def _closure():
        opt_lb.zero_grad()
        loss, lF, lKG = total_loss(model, t_full, m_sim, rho_m0, rho_r0, rho_l0,
                                    w_F=w_F, w_KG=w_KG, causal_eps=causal_eps)
        loss.backward(); lb_buf.append(loss.item()); return loss

    t0 = time.time(); opt_lb.step(_closure)
    print(f"  L-BFGS done {time.time()-t0:.1f}s  final={lb_buf[-1]:.3e}")
    hist['lbfgs_loss'] = lb_buf
    return hist

## 8. Technique 3 — Sequential Window Training

### กลไก
แบ่ง $[0, T_\mathrm{sim}]$ เป็น $K$ windows สั้นๆ แต่ละ window มี $N_\mathrm{osc} / K$ oscillations:

```
Window 1: τ∈[0,Δt]   IC = (a₀, φ₀, φ̇₀)          → train → extract (a₁,φ₁,φ̇₁) at τ=Δt
Window 2: τ∈[0,Δt]   IC = (a₁, φ₁, φ̇₁)          → train → extract (a₂,φ₂,φ̇₂)
...
```

### ทำไมต้อง extract $\dot\phi$?
เพราะ `FullICFieldNet` enforce ทั้ง $\phi(0)$ และ $\dot\phi(0)$ — ถ้าส่งแค่ $\phi$ แต่ไม่มี $\dot\phi$ จะ error propagate
เราคำนวณ $\dot\phi$ via autograd จาก window ก่อนหน้า

### ข้อดีเหนือ Single-window
- $N_\mathrm{osc}$ ต่อ window = $N_\mathrm{osc,total} / K$ → ง่ายขึ้น $K$ เท่า
- Error ไม่สะสมตลอด $[0,T]$ — แต่ละ window reset จากค่าที่แม่นยำ
- Gradient flow สะอาดกว่า (ไม่ผ่าน domain ยาวๆ)

In [9]:
def _extract_ic_at_end(model: GeneralPINN, dt: float,
                       device=DEVICE, dtype=DTYPE) -> tuple[float, float, float]:
    """
    Extract (a, φ, φ̇) at the end of a window (local time = dt).
    φ̇ is computed via autograd so it matches what the PINN learned.
    """
    t_end_t = torch.tensor([[dt]], dtype=dtype, device=device, requires_grad=True)
    a_end, phi_end = model(t_end_t)
    phi_dot_end = torch.autograd.grad(
        phi_end, t_end_t, torch.ones_like(phi_end), create_graph=False)[0]
    return (a_end.detach().item(),
            phi_end.detach().item(),
            phi_dot_end.detach().item())


def sequential_train(mc: dict, phi0: float, a0: float = 1e-8, phi_dot0: float = 0.0,
                     n_windows: int = 5,
                     n_per_band_phi: int = 20, hidden_phi: int = 128, depth_phi: int = 4,
                     n_per_band_a:   int = 16, hidden_a:   int = 64,  depth_a:   int = 3,
                     n_colloc: int = 3000, batch_size: int = 512,
                     adam_epochs: int = 5000, adam_lr: float = 5e-4,
                     lbfgs_iters: int = 1000,
                     causal_eps: float = 0.5, print_every: int = 5000,
                     rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
                     device=DEVICE, dtype=DTYPE) -> list[dict]:
    """
    Train K sequential windows across [0, t_end_sim].

    Each window uses FullICFieldNet + MultiScaleEmbedding.
    IC for window k+1 is extracted from window k via autograd.

    Returns: list of dicts with keys: model, t_lo, t_hi, history
    """
    t_end_sim = mc['t_end_sim']
    m_sim     = mc['m_sim']
    t_edges   = np.linspace(0.0, t_end_sim, n_windows + 1)

    results = []
    cur_a0, cur_phi0, cur_phidot0 = a0, phi0, phi_dot0

    print(f"\n{'='*62}")
    print(f"  Sequential Training: {n_windows} windows  "
          f"m_sim={m_sim:.3g}  T_sim={t_end_sim:.3e}")
    print(f"  N_osc/window ≈ {m_sim/(2*np.pi)*t_end_sim/n_windows:.1f}")
    print(f"{'='*62}")

    for k in range(n_windows):
        t_lo = t_edges[k]
        t_hi = t_edges[k + 1]
        dt   = t_hi - t_lo

        mc_k = {**mc, 't_end_sim': dt,
                'n_osc': m_sim / (2*np.pi) * dt}

        print(f"\n── Window {k+1}/{n_windows}  t∈[{t_lo:.3e}, {t_hi:.3e}]  "
              f"N_osc≈{mc_k['n_osc']:.2f} ──")
        print(f"   IC: a={cur_a0:.4e}  φ={cur_phi0:.5f}  φ'={cur_phidot0:.4e}")

        model_k = build_window_pinn(
            cur_a0, cur_phi0, cur_phidot0, m_sim,
            n_per_band_a=n_per_band_a, n_per_band_phi=n_per_band_phi,
            hidden_a=hidden_a, hidden_phi=hidden_phi,
            depth_a=depth_a, depth_phi=depth_phi, dtype=dtype,
        ).to(device, dtype=dtype)

        hist_k = train_single(
            model_k, mc_k,
            n_colloc=n_colloc, batch_size=batch_size,
            adam_epochs=adam_epochs, adam_lr=adam_lr,
            lbfgs_iters=lbfgs_iters,
            causal_eps=causal_eps, print_every=print_every,
            rho_m0=rho_m0, rho_r0=rho_r0, rho_l0=rho_l0,
            device=device, dtype=dtype,
        )

        # Extract ICs for next window
        cur_a0, cur_phi0, cur_phidot0 = _extract_ic_at_end(model_k, dt, device, dtype)

        results.append(dict(model=model_k, t_lo=t_lo, t_hi=t_hi,
                            history=hist_k, dt=dt))

    print(f"\n  Sequential training complete. {n_windows} windows trained.")
    return results


def sequential_predict(window_results: list[dict],
                       t_eval_sim: np.ndarray,
                       device=DEVICE, dtype=DTYPE) -> tuple[np.ndarray, np.ndarray]:
    """
    Stitch together predictions from all windows into one continuous trajectory.
    Each point is evaluated on the window that contains it.
    """
    a_pred   = np.full_like(t_eval_sim, np.nan)
    phi_pred = np.full_like(t_eval_sim, np.nan)

    for wr in window_results:
        t_lo, t_hi = wr['t_lo'], wr['t_hi']
        # Include left endpoint on all windows, right endpoint only on last
        if wr is window_results[-1]:
            mask = (t_eval_sim >= t_lo) & (t_eval_sim <= t_hi)
        else:
            mask = (t_eval_sim >= t_lo) & (t_eval_sim < t_hi)
        if mask.sum() == 0:
            continue

        t_local = t_eval_sim[mask] - t_lo   # local time within this window
        t_t = torch.tensor(t_local.reshape(-1, 1), dtype=dtype, device=device)
        with torch.no_grad():
            a_w, phi_w = wr['model'](t_t)
        a_pred[mask]   = a_w.cpu().numpy().flatten()
        phi_pred[mask] = phi_w.cpu().numpy().flatten()

    return a_pred, phi_pred

## 8b. Advanced Optimization Techniques

### Root Cause Analysis ของปัญหาที่ยังไม่แก้ได้

| ปัญหา | สาเหตุ | แก้ด้วย |
|---|---|---|
| Loss ไม่ลด / oscillate | $\nabla L_F$ กับ $\nabla L_{KG}$ ชนกัน (dot < 0) | **PCGrad** |
| Error propagate $t=0\to T$ | Causal weight นิ่มเกินไป | **Strong Causal PINN** |
| High-residual region ไม่ถูก sample | Collocation sparse ที่จุดยาก | **RAR** |
| Network ไม่รู้ oscillation ตั้งแต่แรก | Random init ผิด scale มาก | **WKB Pretraining** |
| Frequency ผิดแม้ pointwise error ต่ำ | MSE loss ไม่ sensitive ต่อ phase error | **Spectral Loss** |
| Mass สูงไม่ converge จาก scratch | Loss landscape rough เกินไป | **Curriculum Learning** |

---

### Strong Causal PINN (Wang et al. 2022)

ใช้ **pointwise weight** ที่ขึ้นกับ cumulative residual ทางซ้าย:
$$w_i = \exp\!\left(-\varepsilon \sum_{j < i} \ell_j \right), \quad \ell_j = \Delta t \cdot R_j^2$$

บังคับให้ network แก้ปัญหา **ซ้ายไปขวา** อย่างเคร่งครัด  
$w_i \approx 0$ จนกว่า cumulative loss ทาง left จะลดแล้ว

### PCGrad (Yu et al. 2020 — Gradient Surgery)

ถ้า $g_F \cdot g_{KG} < 0$ (gradients ขัดแย้ง):
$$g_F^* = g_F - \frac{g_F \cdot g_{KG}}{\|g_{KG}\|^2}g_{KG}$$

ป้องกัน loss หนึ่งไป **undo** สิ่งที่อีก loss เพิ่งเรียน

### RAR (Residual-Adaptive Refinement)

ทุก $K$ epochs: evaluate dense residual grid → เพิ่ม collocations ที่ $|\mathcal{R}|$ top-10%

### WKB Pretraining

Pre-train $A_\text{raw}$ ให้ตรง $A_\text{WKB}\propto a^{-3/2}$ ก่อน PINN training  
ลด epochs ที่ต้องใช้ใน main training ลงได้มาก

### Spectral Loss (FFT Domain)

$$\mathcal{L}_\text{spec}(t, R_{KG}) = \frac{1}{N}\sum_k |\hat{R}_{KG}(f_k)|^2 \cdot (1 + \gamma f_k^2)$$

Term $\gamma f_k^2$ **ให้น้ำหนัก high frequency มากขึ้น** → บังคับให้ PINN ลด residual ที่ความถี่สูงด้วย

### Curriculum Learning (Mass Continuation)

$$m_1 = \sqrt{m_{target}}, \quad m_2 = m_{target}^{0.75}, \quad m_3 = m_{target}$$

Warm-start weight จาก $m_k$ ก่อนเริ่ม $m_{k+1}$ → loss landscape smooth กว่ามาก

In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TECHNIQUE: Strong Causal PINN  (Wang et al. 2022)          ║
# ╚══════════════════════════════════════════════════════════════╝

def strong_causal_weights(t_sorted: torch.Tensor,
                           residuals: torch.Tensor,
                           eps: float = 10.0) -> torch.Tensor:
    """
    Compute pointwise causal weights based on cumulative loss to the left.

    w_i = exp(-eps * sum_{j < i} dt_j * r_j^2)

    t_sorted : (N,1) SORTED ascending collocation times
    residuals: (N,1) scalar residual values at each point
               (use max of |R_F|, |R_KG| for combined weighting)
    eps      : causality strength (larger = more strict left-to-right)

    Returns: (N,1) weights in [0, 1], w_0 = 1
    """
    dt = torch.diff(t_sorted, dim=0)                 # (N-1, 1) time gaps
    r2 = (residuals[:-1]**2).detach()               # (N-1, 1) r² at left points
    cum_loss = torch.cumsum(dt * r2, dim=0)          # (N-1, 1) cumulative loss
    # Prepend zero for the first point (always fully weighted)
    cum_loss = torch.cat([torch.zeros(1, 1, dtype=t_sorted.dtype,
                                       device=t_sorted.device), cum_loss], dim=0)
    return torch.exp(-eps * cum_loss)                # (N, 1)


def total_loss_causal(model: "GeneralPINN",
                      t_sorted: torch.Tensor,
                      m_sim: float,
                      rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
                      w_F: float = 1.0, w_KG: float = 1.0,
                      causal_eps: float = 10.0) -> tuple:
    """
    Physics loss with STRONG causal weighting.
    t_sorted must be sorted ascending.
    """
    R_F, R_KG, H = physics_residuals(model, t_sorted, m_sim,
                                      rho_m0, rho_r0, rho_l0)
    # Use max(|R_F|, |R_KG|) as the residual for causality weight computation
    r_max = torch.maximum(torch.abs(R_F.detach()), torch.abs(R_KG.detach()))
    cw = strong_causal_weights(t_sorted, r_max, eps=causal_eps)

    lF  = w_F  * torch.mean(cw * R_F**2)
    lKG = w_KG * torch.mean(cw * R_KG**2)
    return lF + lKG, lF.item(), lKG.item()


# ╔══════════════════════════════════════════════════════════════╗
# ║  TECHNIQUE: PCGrad — Gradient Surgery  (Yu et al. 2020)     ║
# ╚══════════════════════════════════════════════════════════════╝

def pcgrad_step(model: "GeneralPINN",
                t: torch.Tensor, m_sim: float,
                rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
                w_F: float = 1.0, w_KG: float = 1.0,
                causal_eps: float = 10.0,
                optimizer: optim.Optimizer = None) -> tuple:
    """
    PCGrad optimizer step.

    1. Compute separate gradients g_F and g_KG for each loss
    2. If g_F · g_KG < 0: project out the conflicting component
    3. Update with g_F* + g_KG*

    Returns: (total_loss, lF, lKG, conflict_ratio)
    """
    params = list(model.parameters())

    # ── Gradient for Friedmann loss ───────────────────────────
    optimizer.zero_grad()
    R_F, R_KG, _ = physics_residuals(model, t, m_sim, rho_m0, rho_r0, rho_l0)
    t_sort_idx = torch.argsort(t[:, 0])
    t_s = t[t_sort_idx]
    r_max = torch.maximum(torch.abs(R_F.detach()), torch.abs(R_KG.detach()))
    cw = strong_causal_weights(t_s,
                                torch.maximum(torch.abs(R_F[t_sort_idx].detach()),
                                              torch.abs(R_KG[t_sort_idx].detach())),
                                eps=causal_eps)
    # Unsort
    cw_unsorted = torch.zeros_like(cw)
    cw_unsorted[t_sort_idx] = cw

    lF_t  = w_F  * torch.mean(cw_unsorted * R_F**2)
    lF_t.backward(retain_graph=True)
    g_F = [p.grad.clone() if p.grad is not None else torch.zeros_like(p)
           for p in params]

    # ── Gradient for Klein-Gordon loss ────────────────────────
    optimizer.zero_grad()
    R_F2, R_KG2, _ = physics_residuals(model, t, m_sim, rho_m0, rho_r0, rho_l0)
    lKG_t = w_KG * torch.mean(cw_unsorted * R_KG2**2)
    lKG_t.backward()
    g_KG = [p.grad.clone() if p.grad is not None else torch.zeros_like(p)
            for p in params]

    # ── PCGrad: project out conflicting components ────────────
    n_conflict = 0
    g_F_pc  = []
    g_KG_pc = []
    for gf, gkg in zip(g_F, g_KG):
        gf_flat  = gf.view(-1)
        gkg_flat = gkg.view(-1)
        dot = (gf_flat * gkg_flat).sum()
        if dot < 0:  # conflict!
            n_conflict += 1
            # Project g_F to remove component parallel to g_KG
            gf_flat  = gf_flat  - dot / (gkg_flat.norm()**2 + EPS) * gkg_flat
            # Project g_KG to remove component parallel to g_F (original g_F)
            gkg_flat = gkg_flat - dot / (gf.view(-1).norm()**2 + EPS) * gf.view(-1)
        g_F_pc.append(gf_flat.view_as(gf))
        g_KG_pc.append(gkg_flat.view_as(gkg))

    conflict_ratio = n_conflict / max(len(params), 1)

    # ── Apply combined gradient ───────────────────────────────
    optimizer.zero_grad()
    for p, gf, gkg in zip(params, g_F_pc, g_KG_pc):
        p.grad = gf + gkg

    nn.utils.clip_grad_norm_(params, max_norm=1.0)
    optimizer.step()

    lF_val  = lF_t.item()
    lKG_val = lKG_t.item()
    return lF_val + lKG_val, lF_val, lKG_val, conflict_ratio


# ╔══════════════════════════════════════════════════════════════╗
# ║  TECHNIQUE: RAR — Residual Adaptive Refinement              ║
# ╚══════════════════════════════════════════════════════════════╝

def rar_add_points(model: "GeneralPINN",
                   current_t: torch.Tensor,
                   m_sim: float, t_end_sim: float,
                   n_add: int = 200, top_frac: float = 0.10,
                   rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
                   device=DEVICE, dtype=DTYPE) -> torch.Tensor:
    """
    RAR: Evaluate residual on dense grid, add n_add points from high-residual regions.

    Samples n_add new points proportional to |R_KG|^2 + |R_F|^2 (CDF inverse sampling).
    Returns updated collocation tensor.
    """
    model.eval()
    # Dense evaluation grid
    t_dense = torch.linspace(1e-10, t_end_sim, 2000, dtype=dtype, device=device).reshape(-1, 1)

    with torch.no_grad():
        # Quick residual (no create_graph needed)
        t_d = t_dense.clone().requires_grad_(True)
        R_F, R_KG, _ = physics_residuals(model, t_d, m_sim, rho_m0, rho_r0, rho_l0)
        residuals = (R_F**2 + R_KG**2).detach().flatten()

    # Sample from distribution proportional to residual magnitude
    probs = (residuals / (residuals.sum() + EPS)).cpu().numpy()
    idx_new = np.random.choice(len(probs), size=n_add, replace=True, p=probs)
    t_new = t_dense[idx_new].detach()

    # Concatenate and sort
    t_updated = torch.cat([current_t, t_new], dim=0)
    sort_idx  = torch.argsort(t_updated[:, 0])
    model.train()
    return t_updated[sort_idx]


# ╔══════════════════════════════════════════════════════════════╗
# ║  TECHNIQUE: Spectral Loss (FFT domain)                      ║
# ╚══════════════════════════════════════════════════════════════╝

def spectral_loss(model: "GeneralPINN",
                  t_uniform: torch.Tensor,
                  m_sim: float,
                  rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
                  freq_weight_gamma: float = 0.5) -> torch.Tensor:
    """
    Spectral loss: penalize Fourier spectrum of the KG residual.

    Uses uniform t_uniform grid (required for proper FFT).
    Weight: (1 + γ * f²) → amplifies high-frequency mismatches.

    Returns scalar loss.
    """
    t_u = t_uniform.clone().requires_grad_(True)
    R_F, R_KG, _ = physics_residuals(model, t_u, m_sim, rho_m0, rho_r0, rho_l0)

    r = R_KG.squeeze()   # (N,)
    N = r.shape[0]

    # FFT of the residual
    R_hat = torch.fft.rfft(r)    # complex, shape (N//2+1,)
    freqs = torch.fft.rfftfreq(N, dtype=r.dtype, device=r.device)  # normalised freqs

    # Frequency-weighted spectral power
    weights = 1.0 + freq_weight_gamma * (freqs * m_sim)**2
    spec_loss = torch.mean(weights * torch.abs(R_hat)**2) / (N**2)
    return spec_loss


# ╔══════════════════════════════════════════════════════════════╗
# ║  TECHNIQUE: WKB Analytic Pretraining                        ║
# ╚══════════════════════════════════════════════════════════════╝

def wkb_pretrain(model: "GeneralPINN",
                 ode_sol: dict,
                 mc: dict,
                 phi0: float,
                 n_epochs: int = 500,
                 lr: float = 1e-3,
                 device=DEVICE, dtype=DTYPE) -> list[float]:
    """
    Pre-train WKBFieldNet.A(t) to match the analytic WKB envelope:
        A_WKB(t) = phi0 * (a0 / a(t))^{3/2}

    This gives the network a physically-correct starting point for the envelope
    before PINN training begins. Only trains the phi_net (not the a_net).

    Returns: list of pretrain losses.
    """
    if not isinstance(model.phi_net, WKBFieldNet):
        print("  WKB pretraining skipped (not a WKBFieldNet).")
        return []

    model.train()
    t_ref = torch.tensor(ode_sol['t'].reshape(-1, 1), dtype=dtype, device=device)
    a_ref = torch.tensor(ode_sol['a'].reshape(-1, 1), dtype=dtype, device=device)
    a0    = float(ode_sol['a'][0])

    # Analytic WKB target for A(t)
    A_target = phi0 * (a0 / (a_ref + EPS))**1.5    # (N, 1)

    # Only train phi_net parameters
    opt_pre = optim.Adam(model.phi_net.parameters(), lr=lr, eps=1e-9)
    losses  = []

    for epoch in range(n_epochs):
        opt_pre.zero_grad()
        A_pred, _ = model.phi_net.get_envelope_phase(t_ref)
        loss = torch.mean((A_pred - A_target.detach())**2)
        loss.backward()
        nn.utils.clip_grad_norm_(model.phi_net.parameters(), 1.0)
        opt_pre.step()
        losses.append(loss.item())

    final = losses[-1]
    print(f"  WKB pretrain done ({n_epochs} epochs)  final loss = {final:.3e}")
    return losses


print("Advanced optimization tools loaded:")
print("  ✓ strong_causal_weights  — pointwise causal weighting")
print("  ✓ total_loss_causal      — loss with strong causal")
print("  ✓ pcgrad_step            — gradient surgery optimizer step")
print("  ✓ rar_add_points         — residual-adaptive collocation")
print("  ✓ spectral_loss          — FFT domain loss")
print("  ✓ wkb_pretrain           — analytic envelope pretraining")

Advanced optimization tools loaded:
  ✓ strong_causal_weights  — pointwise causal weighting
  ✓ total_loss_causal      — loss with strong causal
  ✓ pcgrad_step            — gradient surgery optimizer step
  ✓ rar_add_points         — residual-adaptive collocation
  ✓ spectral_loss          — FFT domain loss
  ✓ wkb_pretrain           — analytic envelope pretraining


In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  COMBINED: train_advanced() — All Techniques Together       ║
# ╚══════════════════════════════════════════════════════════════╝

def train_advanced(
    model: "GeneralPINN",
    mc: dict,
    ode_sol: dict,            # needed for WKB pretraining
    phi0: float,
    # ── Training sizes ──────────────────────────────────────────
    n_colloc: int   = 4000,
    batch_size: int = 512,
    # ── Adam phase ──────────────────────────────────────────────
    adam_epochs: int   = 10000,
    adam_lr: float     = 3e-4,
    # ── L-BFGS phase ────────────────────────────────────────────
    lbfgs_iters: int   = 3000,
    # ── WKB pretraining ─────────────────────────────────────────
    pretrain_epochs: int = 500,
    pretrain_lr: float   = 1e-3,
    # ── Causal ──────────────────────────────────────────────────
    causal_eps: float  = 10.0,   # strong (vs 0.5 soft)
    # ── PCGrad ──────────────────────────────────────────────────
    use_pcgrad: bool   = True,
    # ── RAR ─────────────────────────────────────────────────────
    use_rar: bool      = True,
    rar_every: int     = 1000,   # add RAR points every N epochs
    rar_add: int       = 200,    # how many points to add each RAR step
    rar_max: int       = 8000,   # cap total collocation points
    # ── Spectral loss ────────────────────────────────────────────
    use_spectral: bool = True,
    spectral_weight: float = 0.05,   # λ_spec in total loss
    spectral_gamma: float  = 0.5,
    # ── Adaptive weights ────────────────────────────────────────
    print_every: int   = 1000,
    rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
    device=DEVICE, dtype=DTYPE,
) -> dict:
    """
    Advanced training pipeline combining ALL techniques:

    1. WKB Pretraining      — analytic envelope initialization
    2. Adam + Strong Causal — pointwise causality enforcement
    3. PCGrad               — gradient surgery on F vs KG conflict
    4. RAR                  — adaptive collocation refinement
    5. Spectral Loss        — FFT domain residual penalty
    6. L-BFGS               — second-order refinement

    All are optional via use_* flags.
    """
    m_sim     = mc['m_sim']
    t_end_sim = mc['t_end_sim']
    a_osc     = mc['a_osc']

    hist = dict(
        adam_loss=[], lF=[], lKG=[], lSpec=[],
        w_F=[], w_KG=[], conflict_ratio=[],
        n_colloc_history=[],
        pretrain_loss=[],
        lbfgs_loss=[],
    )

    # ── Phase 0: WKB Pretraining ──────────────────────────────
    if pretrain_epochs > 0:
        print("\n── Phase 0: WKB Pretraining ──────────────────────────")
        pre_losses = wkb_pretrain(model, ode_sol, mc, phi0,
                                  n_epochs=pretrain_epochs, lr=pretrain_lr,
                                  device=device, dtype=dtype)
        hist['pretrain_loss'] = pre_losses

    # ── Phase 1: Adam (Strong Causal + PCGrad + RAR + Spectral) ─
    aw = AdaptiveWeights(update_every=200)

    opt = optim.Adam(model.parameters(), lr=adam_lr, eps=1e-9,
                     betas=(0.9, 0.999), weight_decay=0)
    sched = CosineAnnealingLR(opt, T_max=adam_epochs, eta_min=adam_lr * 1e-3)

    print(f"\n── Phase 1: Adam ({adam_epochs} epochs) ──────────────────────")
    print(f"   PCGrad={use_pcgrad}  RAR={use_rar}  Spectral={use_spectral}  "
          f"causal_eps={causal_eps}")

    # Initial collocation (sorted for causal weights)
    t_pool = _sample_collocation(n_colloc, 0.0, t_end_sim, a_osc, dtype, device)
    # Sort ascending
    t_pool = t_pool[torch.argsort(t_pool[:, 0])]

    t0 = time.time()
    for epoch in range(adam_epochs):
        # Sub-sample batch (preserve sort order)
        idx  = torch.sort(torch.randperm(len(t_pool), device=device)[:batch_size])[0]
        t_b  = t_pool[idx].clone()

        w_F, w_KG = aw.get()

        if use_pcgrad:
            # PCGrad handles backward + step internally
            loss_val, lF, lKG, cr = pcgrad_step(
                model, t_b, m_sim, rho_m0, rho_r0, rho_l0,
                w_F=w_F, w_KG=w_KG, causal_eps=causal_eps, optimizer=opt)
        else:
            opt.zero_grad()
            loss, lF, lKG = total_loss_causal(
                model, t_b, m_sim, rho_m0, rho_r0, rho_l0,
                w_F=w_F, w_KG=w_KG, causal_eps=causal_eps)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_val = loss.item()
            cr = 0.0

        # Optional spectral loss (uniform grid, separate backward)
        l_spec = 0.0
        if use_spectral and epoch % 5 == 0:   # every 5 epochs (cost control)
            n_spec = min(512, len(t_pool))
            t_uni  = torch.linspace(1e-10, t_end_sim, n_spec,
                                    dtype=dtype, device=device).reshape(-1, 1)
            opt.zero_grad()
            sl = spectral_weight * spectral_loss(
                model, t_uni, m_sim, rho_m0, rho_r0, rho_l0,
                freq_weight_gamma=spectral_gamma)
            sl.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            opt.step()
            l_spec = sl.item()

        sched.step()
        aw.step(lF, lKG)

        hist['adam_loss'].append(loss_val)
        hist['lF'].append(lF); hist['lKG'].append(lKG)
        hist['lSpec'].append(l_spec)
        hist['w_F'].append(w_F); hist['w_KG'].append(w_KG)
        hist['conflict_ratio'].append(cr)
        hist['n_colloc_history'].append(len(t_pool))

        # RAR: add points in high-residual areas
        if use_rar and epoch > 0 and epoch % rar_every == 0:
            if len(t_pool) < rar_max:
                t_pool = rar_add_points(
                    model, t_pool, m_sim, t_end_sim,
                    n_add=rar_add, rho_m0=rho_m0, rho_r0=rho_r0, rho_l0=rho_l0,
                    device=device, dtype=dtype)
                t_pool = t_pool[torch.argsort(t_pool[:, 0])]

        if epoch % print_every == 0 or epoch == adam_epochs - 1:
            lr_now = opt.param_groups[0]['lr']
            avg_cr = float(np.mean(hist['conflict_ratio'][-100:])) if hist['conflict_ratio'] else 0
            print(f"  [{epoch:5d}/{adam_epochs}] "
                  f"loss={loss_val:.3e}  lF={lF:.3e}  lKG={lKG:.3e}  "
                  f"spec={l_spec:.2e}  conflict={avg_cr:.2f}  "
                  f"n_col={len(t_pool)}  lr={lr_now:.2e}")

    print(f"  Adam done in {time.time()-t0:.1f}s")

    # ── Phase 2: L-BFGS ───────────────────────────────────────
    print(f"\n── Phase 2: L-BFGS ({lbfgs_iters} iterations) ──────────────────")
    t_full = t_pool.clone().requires_grad_(True)
    w_F, w_KG = aw.get()
    lb_buf: list[float] = []

    opt_lb = optim.LBFGS(
        model.parameters(), lr=1.0, max_iter=lbfgs_iters,
        tolerance_grad=1e-11, tolerance_change=1e-12,
        history_size=200, line_search_fn='strong_wolfe')

    def _lb_closure():
        opt_lb.zero_grad()
        loss, lF, lKG = total_loss_causal(
            model, t_full, m_sim, rho_m0, rho_r0, rho_l0,
            w_F=w_F, w_KG=w_KG, causal_eps=causal_eps)
        loss.backward(); lb_buf.append(loss.item()); return loss

    t0 = time.time()
    opt_lb.step(_lb_closure)
    hist['lbfgs_loss'] = lb_buf
    print(f"  L-BFGS done in {time.time()-t0:.1f}s  final={lb_buf[-1]:.3e}")

    return hist


# ╔══════════════════════════════════════════════════════════════╗
# ║  TECHNIQUE: Curriculum Learning (Mass Continuation)         ║
# ╚══════════════════════════════════════════════════════════════╝

def curriculum_train(
    m_target: float,
    phi0: float, a0: float = 1e-8, t_end: float = 1.0,
    # Curriculum schedule: list of (m_ratio, epochs, lr)
    # m_ratio is multiplied to m_target to get each stage's effective m
    curriculum_schedule: list = None,
    # Network sizes
    n_per_band_a: int = 16, n_per_band_phi: int = 20,
    hidden_a: int = 64,  hidden_phi: int = 128,
    depth_a: int = 3,    depth_phi: int = 3,
    # Advanced flags
    use_pcgrad: bool = True, use_rar: bool = True,
    use_spectral: bool = True,
    pretrain_epochs: int = 300,
    causal_eps: float = 10.0,
    lbfgs_iters_final: int = 3000,
    n_colloc: int = 4000, batch_size: int = 512,
    print_every: int = 2000,
    rho_m0=RHO_M0, rho_r0=RHO_R0, rho_l0=RHO_L0,
    device=DEVICE, dtype=DTYPE,
) -> dict:
    """
    Curriculum Learning: train from easy (low mass) to hard (target mass).

    Default schedule: m_target^{1/3} → m_target^{2/3} → m_target
    Each stage warm-starts from previous stage.

    Returns: dict with model, all histories, mc for target mass.
    """
    if curriculum_schedule is None:
        # 3-stage: cubic root spacing in log space
        m_stages = [
            m_target**(1.0/3.0),     # stage 1: very easy
            m_target**(2.0/3.0),     # stage 2: medium
            m_target,                 # stage 3: target
        ]
        epochs_per_stage = [5000, 6000, 8000]
        lr_per_stage     = [5e-4, 4e-4, 3e-4]
    else:
        m_stages, epochs_per_stage, lr_per_stage = zip(*curriculum_schedule)

    print(f"\n{'='*65}")
    print(f"  CURRICULUM LEARNING:  m_target={m_target:.3g}")
    print(f"  Stages: {[f'{m:.2g}' for m in m_stages]}")
    print(f"{'='*65}")

    model    = None
    all_hist = []
    mc_final = None

    for stage_i, (m_s, n_ep, lr_s) in enumerate(zip(m_stages, epochs_per_stage, lr_per_stage)):
        print(f"\n{'─'*65}")
        print(f"  STAGE {stage_i+1}/{len(m_stages)}:  m = {m_s:.4g}  epochs={n_ep}  lr={lr_s}")
        print(f"{'─'*65}")

        mc_s = analyze_mass_regime(m_s, t_end)

        # Solve ODE at this mass
        ode_s = solve_ode(mc_s['m_sim'], phi0, 0.0, a0, mc_s['t_end_sim'])

        # Build model at first stage; reuse (with adapted freq) for later stages
        if model is None:
            model = build_advanced_pinn(mc_s, phi0, 0.0, a0,
                                        n_per_band_a, n_per_band_phi,
                                        hidden_a, hidden_phi, depth_a, depth_phi,
                                        dtype=dtype).to(device, dtype=dtype)
            n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Model built: {n_p:,} parameters")
        else:
            # Update WKBFieldNet's m_sim for this stage (important for the analytic part)
            model.phi_net.m_sim = float(mc_s['m_sim'])
            print(f"  Warm-starting from previous stage  (phi_net.m_sim → {mc_s['m_sim']:.3g})")

        hist_s = train_advanced(
            model, mc_s, ode_s, phi0,
            n_colloc=n_colloc, batch_size=batch_size,
            adam_epochs=n_ep, adam_lr=lr_s,
            lbfgs_iters=500,           # short L-BFGS per stage (save for final)
            pretrain_epochs=pretrain_epochs if stage_i == 0 else 0,
            causal_eps=causal_eps,
            use_pcgrad=use_pcgrad, use_rar=use_rar,
            use_spectral=use_spectral,
            print_every=print_every,
            rho_m0=rho_m0, rho_r0=rho_r0, rho_l0=rho_l0,
            device=device, dtype=dtype,
        )
        all_hist.append(hist_s)
        mc_final = mc_s

    # Final L-BFGS at target mass
    mc_target = analyze_mass_regime(m_target, t_end)
    ode_target = solve_ode(mc_target['m_sim'], phi0, 0.0, a0, mc_target['t_end_sim'])
    model.phi_net.m_sim = float(mc_target['m_sim'])

    print(f"\n── Final L-BFGS at m_target={m_target:.3g} ({lbfgs_iters_final} iters) ──")
    t_full = _sample_collocation(n_colloc, 0.0, mc_target['t_end_sim'],
                                  mc_target['a_osc'], dtype, device)
    t_full = t_full[torch.argsort(t_full[:, 0])].requires_grad_(True)
    lb_buf: list[float] = []
    opt_lb = optim.LBFGS(model.parameters(), lr=1.0, max_iter=lbfgs_iters_final,
                          tolerance_grad=1e-11, tolerance_change=1e-12,
                          history_size=200, line_search_fn='strong_wolfe')

    def _final_closure():
        opt_lb.zero_grad()
        loss, _, _ = total_loss_causal(model, t_full, mc_target['m_sim'],
                                        rho_m0, rho_r0, rho_l0,
                                        causal_eps=causal_eps)
        loss.backward(); lb_buf.append(loss.item()); return loss

    opt_lb.step(_final_closure)
    print(f"  Final L-BFGS done  final={lb_buf[-1]:.3e}")

    return dict(model=model, mc=mc_target, ode_sol=ode_target,
                all_histories=all_hist, final_lbfgs_loss=lb_buf)


print("✓ train_advanced() loaded  — all techniques combined")
print("✓ curriculum_train() loaded — mass continuation")

✓ train_advanced() loaded  — all techniques combined
✓ curriculum_train() loaded — mass continuation


## 9. Evaluation, Metrics & Visualization

In [12]:
def _l2rel(pred, ref):
    return np.linalg.norm(pred - ref) / (np.linalg.norm(ref) + EPS)


def compute_metrics(a_pred, phi_pred, a_ode, phi_ode, label='') -> dict:
    """L2 relative errors + max absolute errors."""
    m = dict(
        label   = label,
        L2_a    = _l2rel(a_pred, a_ode),
        L2_phi  = _l2rel(phi_pred, phi_ode),
        max_a   = np.max(np.abs(a_pred - a_ode)),
        max_phi = np.max(np.abs(phi_pred - phi_ode)),
    )
    print(f"  [{label}]  L2(a)={m['L2_a']:.3e}  L2(φ)={m['L2_phi']:.3e}  "
          f"max(a)={m['max_a']:.3e}  max(φ)={m['max_phi']:.3e}")
    return m


def evaluate_wkb(model: GeneralPINN, ode_sol: dict, mc: dict,
                 n_eval: int = 5000, device=DEVICE, dtype=DTYPE) -> dict:
    """Evaluate AdvancedPINN (WKB) model and compare with ODE."""
    model.eval()
    t_sim  = np.logspace(np.log10(max(1e-13, mc['t_end_sim']*1e-5)),
                         np.log10(mc['t_end_sim']), n_eval)
    t_phys = t_sim * mc['lam']

    t_t = torch.tensor(t_sim.reshape(-1, 1), dtype=dtype, device=device)
    with torch.no_grad():
        a_pred   = model.a_net(t_t).cpu().numpy().flatten()
        phi_pred = model.phi_net(t_t).cpu().numpy().flatten()

    a_ode   = interp1d(ode_sol['t'], ode_sol['a'],   kind='cubic',
                       fill_value='extrapolate')(t_sim)
    phi_ode = interp1d(ode_sol['t'], ode_sol['phi'], kind='cubic',
                       fill_value='extrapolate')(t_sim)

    metrics = compute_metrics(a_pred, phi_pred, a_ode, phi_ode, label='WKB')

    # Envelope diagnostics (WKBFieldNet only)
    if isinstance(model.phi_net, WKBFieldNet):
        with torch.no_grad():
            A_arr, dPhi_arr = model.phi_net.get_envelope_phase(t_t)
            A_arr    = A_arr.cpu().numpy().flatten()
            dPhi_arr = dPhi_arr.cpu().numpy().flatten()
    else:
        A_arr = dPhi_arr = None

    return dict(t_sim=t_sim, t_phys=t_phys,
                a_pred=a_pred, phi_pred=phi_pred,
                a_ode=a_ode,   phi_ode=phi_ode,
                A=A_arr, dPhi=dPhi_arr,
                metrics=metrics)


def evaluate_sequential(window_results: list[dict], ode_sol: dict, mc: dict,
                         n_eval: int = 5000, device=DEVICE, dtype=DTYPE) -> dict:
    """Evaluate sequential PINN and compare with ODE."""
    t_sim  = np.logspace(np.log10(max(1e-13, mc['t_end_sim']*1e-5)),
                         np.log10(mc['t_end_sim']), n_eval)
    t_phys = t_sim * mc['lam']

    a_pred, phi_pred = sequential_predict(window_results, t_sim, device, dtype)

    a_ode   = interp1d(ode_sol['t'], ode_sol['a'],   kind='cubic',
                       fill_value='extrapolate')(t_sim)
    phi_ode = interp1d(ode_sol['t'], ode_sol['phi'], kind='cubic',
                       fill_value='extrapolate')(t_sim)

    metrics = compute_metrics(a_pred, phi_pred, a_ode, phi_ode, label='Sequential')
    return dict(t_sim=t_sim, t_phys=t_phys,
                a_pred=a_pred, phi_pred=phi_pred,
                a_ode=a_ode,   phi_ode=phi_ode,
                metrics=metrics)


def plot_comparison(results_dict: dict, mc: dict,
                    save_dir: str = 'results_advanced'):
    """
    Side-by-side comparison of multiple methods vs ODE.
    results_dict: {label: eval_dict}
    """
    os.makedirs(save_dir, exist_ok=True)
    n_methods = len(results_dict)
    colors = ['C1', 'C0', 'C2', 'C3']
    styles = ['-', '--', '-.', ':']

    # Use first result as reference for ODE
    first_ev = next(iter(results_dict.values()))
    t_phys = first_ev['t_phys']
    a_ode  = first_ev['a_ode']
    phi_ode = first_ev['phi_ode']

    fig = plt.figure(figsize=(17, 12), dpi=110)
    gs  = gridspec.GridSpec(3, 3, hspace=0.45, wspace=0.35)

    # ── Row 1: a(t), a error, φ(t) ──────────────────────────
    for ax_idx, (qty, ylabel, title_base) in enumerate(
        [('a_pred', 'a(t)',   'Scale Factor'),
         (None,     'Rel. Err (%)', 'a(t) Error'),
         ('phi_pred', 'φ(t)', 'Axion Field')]):

        ax = fig.add_subplot(gs[0, ax_idx])
        ax.loglog(t_phys, a_ode, 'k-', lw=2, label='ODE',
                  zorder=10) if ax_idx == 0 else None
        if ax_idx == 2:
            ax.semilogx(t_phys, phi_ode, 'k-', lw=1.8, label='ODE', zorder=10)

        for i, (label, ev) in enumerate(results_dict.items()):
            c, s = colors[i], styles[i]
            if ax_idx == 0:
                ax.loglog(ev['t_phys'], ev['a_pred'], color=c, ls=s,
                          lw=1.5, label=label, alpha=0.85)
            elif ax_idx == 1:
                err = np.abs(ev['a_pred'] - ev['a_ode']) / (np.abs(ev['a_ode']) + EPS) * 100
                ax.semilogx(ev['t_phys'], err, color=c, ls=s, lw=1.2, label=label)
            else:
                # show only early portion (first 10% where osc visible)
                mask = t_phys < t_phys[max(1, len(t_phys)//8)]
                ax.semilogx(ev['t_phys'][mask], ev['phi_pred'][mask],
                            color=c, ls=s, lw=1.2, label=label, alpha=0.85)

        if ax_idx == 1:
            ax.axhline(1.0, color='grey', ls=':', lw=1)
        if ax_idx == 2:
            mask = t_phys < t_phys[max(1, len(t_phys)//8)]
            ax.semilogx(t_phys[mask], phi_ode[mask], 'k-', lw=1.8, label='ODE', zorder=10)
        ax.set_xlabel('t (physical)')
        ax.set_ylabel(ylabel)
        ax.set_title(title_base)
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Row 2: φ error, WKB envelope A(t), δΦ(t) ────────────
    ax = fig.add_subplot(gs[1, 0])
    for i, (label, ev) in enumerate(results_dict.items()):
        err = np.abs(ev['phi_pred'] - ev['phi_ode']) / (np.abs(ev['phi_ode']) + EPS) * 100
        ax.semilogx(ev['t_phys'], err + EPS, color=colors[i], ls=styles[i],
                    lw=1.2, label=label)
    ax.axhline(1.0, color='grey', ls=':', lw=1)
    ax.set(xlabel='t (physical)', ylabel='Rel. Err (%)', title='φ(t) Error')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # WKB envelope A(t) (if available)
    ax = fig.add_subplot(gs[1, 1])
    plotted_env = False
    for label, ev in results_dict.items():
        if ev.get('A') is not None:
            ax.semilogx(ev['t_phys'], np.abs(ev['A']), label=rf'$A(t)$ [{label}]', lw=1.5)
            plotted_env = True
    # Analytic WKB prediction: A ∝ a^{-3/2}
    if plotted_env:
        a_ref = a_ode
        A_wkb = np.abs(phi_ode[0]) * (a_ref[0] / (a_ref + EPS))**1.5
        ax.semilogx(t_phys, A_wkb, 'k:', lw=1, label=r'$\propto a^{-3/2}$ (WKB theory)')
    ax.set(xlabel='t (physical)', ylabel='A(t)', title='WKB Envelope Amplitude')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # δΦ(t) phase correction
    ax = fig.add_subplot(gs[1, 2])
    for label, ev in results_dict.items():
        if ev.get('dPhi') is not None:
            ax.semilogx(ev['t_phys'], ev['dPhi'], label=rf'$\delta\Phi(t)$ [{label}]', lw=1.5)
    ax.set(xlabel='t (physical)', ylabel=r'$\delta\Phi(t)$', title='Phase Correction')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Row 3: loss curves + metrics table ───────────────────
    ax = fig.add_subplot(gs[2, 0])
    ax.set_title('Training Loss (Adam)'); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
    for i, (label, ev) in enumerate(results_dict.items()):
        if 'history' in ev:
            ax.semilogy(ev['history']['adam_loss'], color=colors[i], lw=1.2, label=label)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = fig.add_subplot(gs[2, 1])
    ax.set_title('L-BFGS Loss'); ax.set_xlabel('iteration'); ax.set_ylabel('loss')
    for i, (label, ev) in enumerate(results_dict.items()):
        if 'history' in ev and ev['history'].get('lbfgs_loss'):
            ax.semilogy(ev['history']['lbfgs_loss'], color=colors[i], lw=1.2, label=label)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Metrics table
    ax = fig.add_subplot(gs[2, 2])
    lines = [
        f"m_target = {mc['m_target']:.3g}  ({mc['regime']})",
        f"m_a ≈ {mc['m_target']*H0_eV:.3e} eV",
        "",
    ]
    for label, ev in results_dict.items():
        m = ev['metrics']
        lines += [f"── {label} ──",
                  f"  L2(a)  = {m['L2_a']:.3e}",
                  f"  L2(φ)  = {m['L2_phi']:.3e}",
                  f"  max(a) = {m['max_a']:.3e}", ""]
    ax.text(0.05, 0.97, '\n'.join(lines), ha='left', va='top',
            transform=ax.transAxes, fontfamily='monospace', fontsize=8.5,
            bbox=dict(boxstyle='round', facecolor='#f7f7e8', alpha=0.85))
    ax.set_axis_off(); ax.set_title('Metrics Summary')

    fig.suptitle(
        f'Advanced PINN Comparison  |  m = {mc["m_target"]:.3g}  '
        f'({mc["m_target"]*H0_eV:.2e} eV)',
        fontsize=13, fontweight='bold')
    tag  = f"m{mc['m_target']:.0e}".replace('+', '').replace('e0', 'e')
    path = os.path.join(save_dir, f'{tag}_advanced.png')
    plt.savefig(path, bbox_inches='tight', dpi=120)
    plt.show()
    print(f"Saved → {path}")


def plot_wkb_diagnostics(ev: dict, mc: dict, save_dir='results_advanced'):
    """Detailed WKB diagnostic: show A(t), δΦ(t) and verify φ = A·cos(mt+δΦ)."""
    if ev.get('A') is None:
        print("No WKB envelope data."); return
    os.makedirs(save_dir, exist_ok=True)

    t_phys   = ev['t_phys']
    A        = ev['A']
    dPhi     = ev['dPhi']
    phi_pred = ev['phi_pred']
    phi_ode  = ev['phi_ode']

    fig, axes = plt.subplots(2, 2, figsize=(14, 9), dpi=110)

    # Envelope A(t)
    axes[0,0].semilogx(t_phys, A, 'C1', lw=1.5, label='PINN A(t)')
    a_ref  = ev['a_ode']
    A_wkb  = np.abs(phi_ode[0]) * (a_ref[0] / (a_ref + EPS))**1.5
    axes[0,0].semilogx(t_phys, A_wkb, 'k:', lw=1.2, label=r'WKB theory $\propto a^{-3/2}$')
    axes[0,0].set(xlabel='t', ylabel='A(t)', title='Envelope Amplitude')
    axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

    # Phase correction δΦ(t)
    axes[0,1].semilogx(t_phys, dPhi, 'C2', lw=1.5)
    axes[0,1].set(xlabel='t', ylabel=r'$\delta\Phi(t)$', title='Phase Correction')
    axes[0,1].grid(alpha=0.3)

    # φ(t) comparison — early time
    mask = t_phys < t_phys[len(t_phys)//6]
    axes[1,0].plot(t_phys[mask], phi_ode[mask],  'k-',  lw=1.8, label='ODE')
    axes[1,0].plot(t_phys[mask], phi_pred[mask], 'C1--', lw=1.2, label='PINN (WKB)')
    axes[1,0].set(xlabel='t', ylabel='φ(t)', title='Field Comparison (early time)')
    axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

    # Relative error
    err = np.abs(phi_pred - phi_ode) / (np.abs(phi_ode) + EPS) * 100
    axes[1,1].semilogx(t_phys, err, 'coral', lw=1)
    axes[1,1].axhline(1.0, color='grey', ls='--', lw=1, label='1%')
    axes[1,1].set(xlabel='t', ylabel='Rel. Error (%)', title='φ Relative Error')
    axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

    plt.suptitle(f'WKB Diagnostics  m={mc["m_target"]:.3g}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    tag  = f"m{mc['m_target']:.0e}".replace('+', '').replace('e0', 'e')
    plt.savefig(os.path.join(save_dir, f'{tag}_wkb_diag.png'), bbox_inches='tight', dpi=120)
    plt.show()

## 10. Master Pipelines

```python
# Simplest usage:
ev_wkb  = run_wkb(m_target=200.0)           # WKB + MultiScale single window
ev_seq  = run_sequential(m_target=200.0)    # Sequential windows
```

In [13]:
def run_wkb(
    m_target: float = 200.0,
    phi0: float = 1.0, a0: float = 1e-8, t_end: float = 1.0,
    # Network
    n_per_band_a: int = 16, n_per_band_phi: int = 20,
    hidden_a: int = 64,  hidden_phi: int = 128,
    depth_a:  int = 3,   depth_phi:  int = 3,
    # Training
    adam_epochs: int = 8000, adam_lr: float = 5e-4,
    lbfgs_iters: int = 2000,
    n_colloc: int = 4000, batch_size: int = 512,
    causal_eps: float = 0.5, print_every: int = 1000,
    save_dir: str = 'results_advanced',
    device=DEVICE, dtype=DTYPE,
) -> dict:
    """
    Full WKB + MultiScale Fourier PINN pipeline.
    Operates in simulation coordinates; reports results in physical coords.
    """
    mc = analyze_mass_regime(m_target, t_end)
    print_mass_summary(mc)

    print(f"\nSolving ODE reference (m_sim={mc['m_sim']:.3g}) ...")
    ode_sol = solve_ode(mc['m_sim'], phi0, 0.0, a0, mc['t_end_sim'])

    # Auto-scale network to regime difficulty
    n_osc = mc['m_sim'] / (2*np.pi) * mc['t_end_sim']
    if n_osc > 50:
        n_per_band_phi = max(n_per_band_phi, 28)
        hidden_phi     = max(hidden_phi, 160)
        depth_phi      = max(depth_phi, 4)

    model = build_advanced_pinn(mc, phi0, 0.0, a0,
                                n_per_band_a, n_per_band_phi,
                                hidden_a, hidden_phi, depth_a, depth_phi,
                                dtype=dtype).to(device, dtype=dtype)
    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nAdvanced PINN (WKB+MultiScale)  params={n_p:,}")
    print(f"WKB embedding m_slow={model.phi_net.emb.n_bands} bands  "
          f"out_dim={model.phi_net.emb.out_dim}")

    history = train_single(model, mc,
                           n_colloc=n_colloc, batch_size=batch_size,
                           adam_epochs=adam_epochs, adam_lr=adam_lr,
                           lbfgs_iters=lbfgs_iters,
                           causal_eps=causal_eps, print_every=print_every,
                           device=device, dtype=dtype)

    ev = evaluate_wkb(model, ode_sol, mc, device=device, dtype=dtype)
    ev['history'] = history

    plot_wkb_diagnostics(ev, mc, save_dir)
    return dict(mc=mc, model=model, ode_sol=ode_sol, history=history, eval=ev)


def run_sequential(
    m_target: float = 200.0,
    phi0: float = 1.0, a0: float = 1e-8, t_end: float = 1.0,
    n_windows: int = 5,
    # Network (per window)
    n_per_band_a:   int = 16, n_per_band_phi: int = 20,
    hidden_a: int = 64,  hidden_phi: int = 128,
    depth_a:  int = 3,   depth_phi:  int = 4,
    # Training (per window)
    adam_epochs: int = 5000, adam_lr: float = 5e-4,
    lbfgs_iters: int = 1000,
    n_colloc: int = 3000, batch_size: int = 512,
    causal_eps: float = 0.5,
    save_dir: str = 'results_advanced',
    device=DEVICE, dtype=DTYPE,
) -> dict:
    """
    Full Sequential Window PINN pipeline.
    Each window: FullICFieldNet + MultiScaleEmbedding.
    """
    mc = analyze_mass_regime(m_target, t_end)
    print_mass_summary(mc)

    print(f"\nSolving ODE reference (m_sim={mc['m_sim']:.3g}) ...")
    ode_sol = solve_ode(mc['m_sim'], phi0, 0.0, a0, mc['t_end_sim'])

    window_results = sequential_train(
        mc, phi0, a0, phi_dot0=0.0,
        n_windows=n_windows,
        n_per_band_a=n_per_band_a, n_per_band_phi=n_per_band_phi,
        hidden_a=hidden_a, hidden_phi=hidden_phi,
        depth_a=depth_a,   depth_phi=depth_phi,
        n_colloc=n_colloc, batch_size=batch_size,
        adam_epochs=adam_epochs, adam_lr=adam_lr,
        lbfgs_iters=lbfgs_iters,
        causal_eps=causal_eps,
        print_every=adam_epochs,   # suppress per-epoch output per window
        device=device, dtype=dtype,
    )

    ev = evaluate_sequential(window_results, ode_sol, mc, device=device, dtype=dtype)
    return dict(mc=mc, ode_sol=ode_sol, window_results=window_results, eval=ev)

## 11. Experiment A — Oscillating Regime: m = 200 H₀

เปรียบเทียบ 3 วิธี:
1. **Baseline** (จาก `Axion_PINN_Single_Field.ipynb`) — standard `SingleAxionPINN` 
2. **WKB** — envelope decomposition + MultiScaleEmbedding (ไฟล์นี้)
3. **Sequential** — 5 windows + FullICFieldNet + MultiScaleEmbedding (ไฟล์นี้)

m=200 อยู่ใน direct mode (N_osc ≈ 32) — เป็น challenge ปานกลางสำหรับ baseline แต่ควรง่ายมากสำหรับ WKB

In [14]:
M_A = 200.0   # code units

# ── WKB run ───────────────────────────────────────────────────
print("=" * 62)
print(f"  EXPERIMENT A:  WKB  m={M_A}")
print("=" * 62)

res_wkb_A = run_wkb(
    m_target    = M_A,
    phi0        = 1.0,
    a0          = 1e-8,
    t_end       = 1.0,
    adam_epochs = 8000,
    adam_lr     = 5e-4,
    lbfgs_iters = 2000,
    n_colloc    = 4000,
    causal_eps  = 0.5,
    print_every = 2000,
)

  EXPERIMENT A:  WKB  m=200.0
────────────────────────────────────────────────────────────
  m_target=200  regime=oscillating  mode=direct
  N_osc(physical)=3.18e+01  |  a_osc=1.194e-02
  m_a ≈ 2.875e-31 eV
────────────────────────────────────────────────────────────

Solving ODE reference (m_sim=200) ...
ODE: 0.40s  success=True  nfev=10856

Advanced PINN (WKB+MultiScale)  params=122,247
WKB embedding m_slow=2 bands  out_dim=80
  Adam  m_sim=200  t_end=1.000e+00  [direct]
    [    0/8000] loss=1.289e+10  lF=8.494e-01  lKG=1.289e+10  lr=5.00e-04


KeyboardInterrupt: 

In [ ]:
# ── Sequential run ────────────────────────────────────────────
print("=" * 62)
print(f"  EXPERIMENT A:  Sequential (5 windows)  m={M_A}")
print("=" * 62)

res_seq_A = run_sequential(
    m_target    = M_A,
    phi0        = 1.0,
    a0          = 1e-8,
    t_end       = 1.0,
    n_windows   = 5,
    adam_epochs = 5000,
    adam_lr     = 5e-4,
    lbfgs_iters = 1000,
    n_colloc    = 3000,
    causal_eps  = 0.5,
)

  EXPERIMENT A:  Sequential (5 windows)  m=200.0
────────────────────────────────────────────────────────────
  m_target=200  regime=oscillating  mode=direct
  N_osc(physical)=3.18e+01  |  a_osc=1.194e-02
  m_a ≈ 2.875e-31 eV
────────────────────────────────────────────────────────────

Solving ODE reference (m_sim=200) ...
ODE: 0.14s  success=True  nfev=10856

  Sequential Training: 5 windows  m_sim=200  T_sim=1.000e+00
  N_osc/window ≈ 6.4

── Window 1/5  t∈[0.000e+00, 2.000e-01]  N_osc≈6.37 ──
   IC: a=1.0000e-08  φ=1.00000  φ'=0.0000e+00
  Adam  m_sim=200  t_end=2.000e-01  [direct]
    [    0/5000] loss=1.280e+01  lF=8.404e-01  lKG=1.196e+01  lr=5.00e-04
    [ 4999/5000] loss=1.098e+00  lF=4.676e-01  lKG=6.308e-01  lr=5.00e-07
  Adam done 134.4s
  L-BFGS refinement ...
  L-BFGS done 3.8s  final=1.110e+00

── Window 2/5  t∈[2.000e-01, 4.000e-01]  N_osc≈6.37 ──
   IC: a=1.0981e+00  φ=0.00297  φ'=6.9309e-01
  Adam  m_sim=200  t_end=2.000e-01  [direct]
    [    0/5000] loss=1.082e+00  

In [ ]:
# ── Baseline run (reimplement SingleAxionPINN inline for comparison) ──
# Uses FullICFieldNet with single-scale Fourier (standard approach)
from torch.optim.lr_scheduler import CosineAnnealingLR as _CALR

class _SingleScaleEmb(nn.Module):
    """Single trainable-scale Fourier (baseline)."""
    def __init__(self, n=64, scale=10.0, dtype=torch.float64):
        super().__init__()
        B = torch.randn(n, 1, dtype=dtype)
        self.register_buffer('B', B)
        self.log_scale = nn.Parameter(torch.tensor(np.log(scale), dtype=dtype))
        self.out_dim = 2 * n
    def forward(self, t):
        s = torch.exp(self.log_scale)
        p = 2*np.pi*s*(t @ self.B.T)
        return torch.cat([torch.sin(p), torch.cos(p)], -1)

class _BaselineFieldNet(nn.Module):
    def __init__(self, phi0, m_sim, n=64, hidden=128, depth=4, dtype=torch.float64):
        super().__init__()
        self.register_buffer('phi0', torch.tensor([[phi0]], dtype=dtype))
        self.emb = _SingleScaleEmb(n, m_sim/(2*np.pi), dtype=dtype)
        layers = [nn.Linear(self.emb.out_dim, hidden, dtype=dtype)]
        for _ in range(depth): layers.append(SinResBlock(hidden, dtype=dtype))
        self.net = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1, dtype=dtype)
    def forward(self, t):
        return self.phi0 + t * self.head(self.net(self.emb(t)))

mc_A = analyze_mass_regime(M_A)
ode_A = solve_ode(mc_A['m_sim'], 1.0, 0.0, 1e-8, mc_A['t_end_sim'])
a_bl = ScaleFactorNet(1e-8, dtype=DTYPE)
p_bl = _BaselineFieldNet(1.0, mc_A['m_sim'], dtype=DTYPE)
model_bl = GeneralPINN(a_bl, p_bl).to(DEVICE, dtype=DTYPE)
hist_bl  = train_single(model_bl, mc_A,
                        adam_epochs=8000, adam_lr=5e-4, lbfgs_iters=2000,
                        n_colloc=4000, print_every=4000,
                        device=DEVICE, dtype=DTYPE)
ev_bl_A = evaluate_wkb(model_bl, ode_A, mc_A)
ev_bl_A['history'] = hist_bl

ODE: 0.14s  success=True  nfev=10856
  Adam  m_sim=200  t_end=1.000e+00  [direct]
    [    0/8000] loss=3.895e+18  lF=8.637e-01  lKG=3.895e+18  lr=5.00e-04
    [ 4000/8000] loss=6.184e+15  lF=5.361e+15  lKG=8.236e+14  lr=2.50e-04
    [ 7999/8000] loss=7.404e+14  lF=7.394e+14  lKG=9.374e+11  lr=5.00e-07
  Adam done 208.3s
  L-BFGS refinement ...
  L-BFGS done 18.6s  final=6.471e+14
  [WKB]  L2(a)=2.550e-01  L2(φ)=5.749e-01  max(a)=1.172e+00  max(φ)=1.072e+00


In [ ]:
# ── Three-way comparison plot ─────────────────────────────────
ev_wkb_A = res_wkb_A['eval']
ev_seq_A  = res_seq_A['eval']

# Align all to same t_phys grid using ODE interpolation
t_grid = ev_wkb_A['t_sim']
a_ode_grid   = interp1d(ode_A['t'], ode_A['a'],   kind='cubic', fill_value='extrapolate')(t_grid)
phi_ode_grid = interp1d(ode_A['t'], ode_A['phi'], kind='cubic', fill_value='extrapolate')(t_grid)

ev_seq_A_aligned = dict(
    t_phys   = ev_wkb_A['t_phys'],
    t_sim    = t_grid,
    a_pred   = interp1d(ev_seq_A['t_sim'], ev_seq_A['a_pred'],   kind='linear', bounds_error=False, fill_value='extrapolate')(t_grid),
    phi_pred = interp1d(ev_seq_A['t_sim'], ev_seq_A['phi_pred'], kind='linear', bounds_error=False, fill_value='extrapolate')(t_grid),
    a_ode    = a_ode_grid,
    phi_ode  = phi_ode_grid,
    A=None, dPhi=None,
    metrics  = compute_metrics(
        interp1d(ev_seq_A['t_sim'], ev_seq_A['a_pred'],   kind='linear', bounds_error=False, fill_value='extrapolate')(t_grid),
        interp1d(ev_seq_A['t_sim'], ev_seq_A['phi_pred'], kind='linear', bounds_error=False, fill_value='extrapolate')(t_grid),
        a_ode_grid, phi_ode_grid, label='Sequential (aligned)'),
)

ev_bl_A_aligned = dict(**ev_bl_A,
    t_phys=ev_wkb_A['t_phys'], t_sim=t_grid,
    a_ode=a_ode_grid, phi_ode=phi_ode_grid,
)

plot_comparison({
    'Baseline (single-scale)': ev_bl_A_aligned,
    'WKB + MultiScale':        ev_wkb_A,
    'Sequential (5 win)':      ev_seq_A_aligned,
}, mc_A, save_dir='results_advanced')

  [Sequential (aligned)]  L2(a)=1.361e+00  L2(φ)=5.283e-01  max(a)=1.123e+00  max(φ)=1.168e+00


TypeError: dict() got multiple values for keyword argument 't_phys'

## 12. Experiment B — High Mass + Rescaling: m = 1×10⁵ H₀

ระบอบ rapid-osc: ถ้าไม่ rescale จะมี N_osc ~ 1.6×10⁴  
หลัง rescale: m_sim ≈ 251, t_end_sim ≈ 0.016, N_osc_sim ≈ 40   → WKB ทำงานได้ดีมาก

เทคนิคที่ใช้: **WKB + Sequential (10 windows)**

In [ ]:
M_B = 1e5
mc_B = analyze_mass_regime(M_B)
print_mass_summary(mc_B)

# ── WKB (single window) ───────────────────────────────────────
print("\n=== WKB single window ===")
res_wkb_B = run_wkb(
    m_target    = M_B,
    phi0        = 1.0,
    a0          = 1e-8,
    t_end       = 1.0,
    adam_epochs = 12000,
    adam_lr     = 3e-4,
    lbfgs_iters = 3000,
    n_colloc    = 5000,
    causal_eps  = 1.0,
    print_every = 4000,
)

In [ ]:
# ── Sequential (10 windows) ───────────────────────────────────
print("\n=== Sequential 10 windows ===")
res_seq_B = run_sequential(
    m_target    = M_B,
    phi0        = 1.0,
    a0          = 1e-8,
    t_end       = 1.0,
    n_windows   = 10,
    adam_epochs = 4000,
    adam_lr     = 5e-4,
    lbfgs_iters = 1000,
    n_colloc    = 3000,
    causal_eps  = 0.8,
)

In [ ]:
# ── Comparison plot for Experiment B ─────────────────────────
ode_B   = res_wkb_B['ode_sol']
ev_wkb_B = res_wkb_B['eval']
ev_seq_B = res_seq_B['eval']

t_grid_B = ev_wkb_B['t_sim']
a_ode_B   = interp1d(ode_B['t'], ode_B['a'],   kind='cubic', fill_value='extrapolate')(t_grid_B)
phi_ode_B = interp1d(ode_B['t'], ode_B['phi'], kind='cubic', fill_value='extrapolate')(t_grid_B)

ev_seq_B_al = dict(
    t_phys=ev_wkb_B['t_phys'], t_sim=t_grid_B,
    a_pred=interp1d(ev_seq_B['t_sim'], ev_seq_B['a_pred'],   bounds_error=False, fill_value='extrapolate')(t_grid_B),
    phi_pred=interp1d(ev_seq_B['t_sim'], ev_seq_B['phi_pred'], bounds_error=False, fill_value='extrapolate')(t_grid_B),
    a_ode=a_ode_B, phi_ode=phi_ode_B, A=None, dPhi=None,
    metrics=compute_metrics(
        interp1d(ev_seq_B['t_sim'], ev_seq_B['a_pred'],   bounds_error=False, fill_value='extrapolate')(t_grid_B),
        interp1d(ev_seq_B['t_sim'], ev_seq_B['phi_pred'], bounds_error=False, fill_value='extrapolate')(t_grid_B),
        a_ode_B, phi_ode_B, label='Seq-10win (aligned)'),
)

plot_comparison({
    'WKB + MultiScale':     ev_wkb_B,
    'Sequential (10 win)':  ev_seq_B_al,
}, mc_B, save_dir='results_advanced')

# WKB specific diagnostics
plot_wkb_diagnostics(ev_wkb_B, mc_B)

## 13. Experiment C — Extreme Mass: m = 1×10⁸ H₀ (WKB เท่านั้นที่ทำได้)

มวลสูงมาก: N_osc_physical ≈ 1.6×10⁷  
หลัง rescaling: m_sim จะถูก auto-chosen, N_osc_sim ≈ 250  
WKB ลด effective oscillations สำหรับ network ให้เหลือ ~0 (แค่เรียน envelope smooth)

In [ ]:
mc_C = analyze_mass_regime(1e8)
print_mass_summary(mc_C)

res_wkb_C = run_wkb(
    m_target    = 1e8,
    phi0        = 1.0,
    a0          = 1e-8,
    t_end       = 1.0,
    # Larger network for extreme case
    n_per_band_phi = 28,
    hidden_phi     = 192,
    depth_phi      = 4,
    adam_epochs    = 15000,
    adam_lr        = 2e-4,
    lbfgs_iters    = 4000,
    n_colloc       = 5000,
    causal_eps     = 1.5,
    print_every    = 5000,
)

plot_wkb_diagnostics(res_wkb_C['eval'], mc_C)

## Experiment D: `train_advanced()` — All Techniques Combined (m = 200)

This experiment runs the **same m=200 case** as Experiment A but uses `train_advanced()` with all six new optimization flags enabled:

| Flag | Effect |
|---|---|
| `pretrain_epochs=500` | WKB envelope pre-training before Adam |
| `causal_eps=10.0` | Strong causal weighting (Wang et al. 2022) |
| `use_pcgrad=True` | PCGrad gradient surgery for F vs KG conflict |
| `use_rar=True` | Residual-Adaptive Refinement every 1000 steps |
| `use_spectral=True` | FFT spectral loss on KG residual |

Compare `err_phi` against the baseline WKB result from Experiment A.


In [ ]:

# ─── Experiment D: train_advanced() for m=200 ─────────────────────────────────
import time

PHI0_D   = 1.0
A0_D     = 1e-8
M_D      = 200.0
T_END_D  = 1.0

print("=" * 65)
print(f"Experiment D: train_advanced() with ALL techniques  (m={M_D:.0f})")
print("=" * 65)

mc_D   = analyze_mass_regime(M_D, t_end=T_END_D)
ode_D  = solve_ode(mc_D['m_sim'], PHI0_D, 0.0, A0_D, mc_D['t_end_sim'])
model_D = build_advanced_pinn(mc_D, PHI0_D, a0=A0_D, hidden=256, layers=6)

t0_D = time.time()
hist_D = train_advanced(
    model_D, mc_D, ode_D, phi0=PHI0_D,
    n_colloc       = 4000,
    adam_epochs    = 12000,
    adam_lr        = 3e-4,
    lbfgs_iters    = 3000,
    pretrain_epochs= 500,
    pretrain_lr    = 1e-3,
    causal_eps     = 10.0,
    use_pcgrad     = True,
    use_rar        = True,
    use_spectral   = True,
    rar_every      = 1000,
    rar_add        = 300,
    rar_max        = 8000,
    spectral_weight= 0.05,
    spectral_gamma = 0.5,
    print_every    = 2000,
)
print(f"Elapsed: {time.time()-t0_D:.1f} s")

ev_D = evaluate_wkb(model_D, ode_D, mc_D)
print("\n--- Experiment D results ---")
for k, v in ev_D.items():
    if k != 'data':
        print(f"  {k:20s} = {v}")

# Plot — use existing plot_comparison helper
try:
    plot_comparison(
        t_phys   = ev_D['data']['t_phys'],
        phi_ode  = ev_D['data']['phi_ode'],
        phi_pinn = ev_D['data']['phi_pinn'],
        a_ode    = ev_D['data']['a_ode'],
        a_pinn   = ev_D['data']['a_pinn'],
        mc       = mc_D,
        title    = f"Experiment D — train_advanced()  m={M_D:.0f}",
    )
except Exception as exc:
    print(f"[plot skipped: {exc}]")


## Experiment E: `curriculum_train()` — Mass Continuation (m = 1e5)

For masses where standard training fails completely, **curriculum training** warms up through three stages:

$$m_1 = m^{1/3}, \quad m_2 = m^{2/3}, \quad m_3 = m \quad (\text{target})$$

Each stage inherits the network weights of the previous stage, so the model is never initialising cold at the full oscillation frequency. All advanced techniques (`PCGrad`, `RAR`, `Spectral`) remain active throughout.


In [ ]:

# ─── Experiment E: curriculum_train() for m=1e5 ───────────────────────────────
import time

PHI0_E  = 1.0
A0_E    = 1e-8
M_E     = 1e5
T_END_E = 1.0

print("=" * 65)
print(f"Experiment E: curriculum_train()  (m={M_E:.0e})")
print("=" * 65)

t0_E = time.time()
res_E = curriculum_train(
    m_target          = M_E,
    phi0              = PHI0_E,
    a0                = A0_E,
    t_end             = T_END_E,
    curriculum_schedule = None,   # uses default m^(1/3) → m^(2/3) → m
    use_pcgrad        = True,
    use_rar           = True,
    use_spectral      = True,
    pretrain_epochs   = 300,
    causal_eps        = 10.0,
    lbfgs_iters_final = 3000,
    print_every       = 2000,
)
print(f"Total elapsed: {time.time()-t0_E:.1f} s")

# Evaluate final stage
ev_E = evaluate_wkb(res_E['model'], res_E['ode_sol'], res_E['mc'])
print("\n--- Experiment E results ---")
for k, v in ev_E.items():
    if k != 'data':
        print(f"  {k:20s} = {v}")

# Plot
try:
    plot_comparison(
        t_phys   = ev_E['data']['t_phys'],
        phi_ode  = ev_E['data']['phi_ode'],
        phi_pinn = ev_E['data']['phi_pinn'],
        a_ode    = ev_E['data']['a_ode'],
        a_pinn   = ev_E['data']['a_pinn'],
        mc       = res_E['mc'],
        title    = f"Experiment E — curriculum_train()  m={M_E:.0e}",
    )
except Exception as exc:
    print(f"[plot skipped: {exc}]")

# ─── Summary comparison ───────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("Summary: new techniques vs baseline (m=200 case)")
print("=" * 55)
try:
    import pandas as pd
    rows = []
    for label, ev in [("Exp D — train_advanced", ev_D),
                      ("Exp E — curriculum (m=1e5)", ev_E)]:
        rows.append({
            'Experiment' : label,
            'err_phi (L2)': f"{ev.get('err_phi', float('nan')):.4f}",
            'err_a  (L2)': f"{ev.get('err_a',   float('nan')):.4f}",
            'res_KG (max)': f"{ev.get('max_res_KG', float('nan')):.2e}",
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
except Exception:
    print(f"  train_advanced  err_phi={ev_D.get('err_phi','?')}")
    print(f"  curriculum      err_phi={ev_E.get('err_phi','?')}")


## 14. Summary Table & Interpretation

In [ ]:
print("=" * 70)
print("  FINAL RESULTS SUMMARY")
print("=" * 70)
print(f"{'Experiment':<30}  {'Method':<22}  {'L2(a)':>9}  {'L2(φ)':>9}")
print("-" * 70)

rows = [
    ("Exp A  m=200", "Baseline (single-scale)",  ev_bl_A_aligned['metrics']['L2_a'],   ev_bl_A_aligned['metrics']['L2_phi']),
    ("Exp A  m=200", "WKB + MultiScale",          ev_wkb_A['metrics']['L2_a'],          ev_wkb_A['metrics']['L2_phi']),
    ("Exp A  m=200", "Sequential 5 win",           ev_seq_A_aligned['metrics']['L2_a'],  ev_seq_A_aligned['metrics']['L2_phi']),
    ("Exp B  m=1e5", "WKB + MultiScale",           ev_wkb_B['metrics']['L2_a'],          ev_wkb_B['metrics']['L2_phi']),
    ("Exp B  m=1e5", "Sequential 10 win",          ev_seq_B_al['metrics']['L2_a'],       ev_seq_B_al['metrics']['L2_phi']),
    ("Exp C  m=1e8", "WKB + MultiScale",           res_wkb_C['eval']['metrics']['L2_a'], res_wkb_C['eval']['metrics']['L2_phi']),
]

for exp, method, La, Lp in rows:
    flag = " ✓" if (La < 0.01 and Lp < 0.1) else ("  " if La < 0.1 else " ✗")
    print(f"  {exp:<28}  {method:<22}  {La:>9.3e}  {Lp:>9.3e}{flag}")

print("=" * 70)
print("\nInterpretation:")
print("  ✓  L2(a)<1%  and  L2(φ)<10%  → acceptable accuracy")
print("  ✗  L2 error large             → needs more training or tuning")
print()
print("Key insight: WKB works because A(t) and δΦ(t) are smooth → network")
print("only needs to learn slowly-varying functions regardless of m.")
print("Sequential windows reduce per-window oscillation count by 1/K.")